# Brain Tumor MRI Classification with Monte Carlo Dropout
## Calibration, Selective Prediction and Uncertainty Analysis — Experiment Notebook

*Code documentation for the journal (Q2) submission.*

## Overview

This notebook contains the complete experimental pipeline: data preparation, hyper-parameter optimisation, the 30-seed final experiment, and all follow-up analyses (statistical tests, MCDA sensitivity, selective prediction, uncertainty decomposition and Grad-CAM).

**Study design (as implemented below)**

| Item | Setting |
|---|---|
| Task | 4-class brain tumor MRI classification: glioma, meningioma, no tumor, pituitary |
| Dataset | Kaggle *Brain Tumor MRI Dataset* (official `Training` / `Testing` folders) |
| Backbones | ResNet18, VGG19, GoogLeNet, MobileNetV2, DenseNet121, AlexNet (ImageNet-pretrained, fine-tuned end-to-end) |
| Uncertainty | Monte Carlo (MC) Dropout: the dropout layer(s) of each backbone are replaced by dropout that stays active at inference |
| Data split | `Training` folder → 80 % train / 20 % validation (stratified, `random_state = 42`); `Testing` folder = test set |
| Augmentation | Offline, training images only (3 augmented copies per image) |
| Hyper-parameter search | Optuna, 25 trials per backbone, objective = validation accuracy |
| Final experiment | 30 independent seeds per backbone (180 runs), MC Dropout with T = 50 stochastic passes |

**Pipeline**

| Section | Purpose | Main outputs |
|---|---|---|
| 0 | Imports, configuration and shared definitions | — |
| 1 | Data preparation: folder scan, stratified split, augmentation | `df_train.csv`, `df_validation.csv`, `df_test.csv`, `augmented_train/` |
| 2 | Optuna hyper-parameter search for the six backbones | `df_report_<model>.csv`, `optuna_6models_combined_report.xlsx` |
| 3.1 | 30-seed experiment, clinical / calibration / rejection metrics, CDSS composite score, seed-normality test, Friedman + post-hoc tests, paired test | `results_30_seeds_<model>.xlsx`, `saved_models/`, `results_180_runs_all_models.xlsx`, `accuracy_normality_analysis.xlsx`, `results_advanced_statistical_tests.xlsx` |
| 3.2 | MCDA sensitivity analysis of the composite ranking | `mcda_sensitivity_results.xlsx`, `mcda_sensitivity.png` |
| 3.3 | Selective prediction and error-detection AUROC over 30 seeds | `uncertainty_cache/`, `results_selective_prediction_30seeds.xlsx` |
| 3.4 | Deterministic vs. standard dropout vs. MC Dropout (ResNet18) | `three_mode_comparison_resnet18.xlsx` |
| 3.5 | Paired Wilcoxon test: deterministic vs. MC Dropout | console table |
| 3.6 | Convergence of MC Dropout with the number of passes T | `mc_dropout_convergence_summary.csv`, `.png` |
| 3.7 | Per-class metrics, confusion matrix, reliability diagram | `plots/*.png` |
| 3.8 | Uncertainty distributions per class and for correct vs. wrong predictions | two `.png` figures |
| 3.9 | Epistemic-uncertainty threshold, MC Grad-CAM, threshold applied to the test set | threshold log, Grad-CAM folders, `.csv`, `.png` |

A complete list of every file, figure and printed table is given in **Appendix A**.

**How to run**

1. Run **Section 0** after every kernel restart (it defines everything the later sections share). Files created by earlier runs under their former Indonesian names are converted once by the cell in Section 0.8.
2. Run **Sections 1 → 2 → 3.1 in order**: they create the CSV / Excel / checkpoint files read by everything else.
3. **Sections 3.2 – 3.9** are independent analyses that only need the files produced by 3.1 (3.3 additionally needs the saved checkpoints). Each of them starts with a short *Inputs / Outputs* note.
4. The three heavy steps are the Optuna search (Section 2), the 180 training runs (Section 3.1) and the MC Grad-CAM export (Section 3.9.2). Sections 2 and 3.1 skip every backbone whose result file already exists (a backbone interrupted halfway restarts from its first seed); Section 3.3 resumes at the seed level through its cache; Section 3.9.2 always regenerates all files.

**Reproducibility notes**

* Train / validation split, shuffling and augmentation use seed 42; the 30 final runs use seeds `101 + 37·i` for `i = 0 … 29` (101 … 1174). Seed 360 (`i = 7`) is the run analysed in detail in Sections 3.7 – 3.9.
* Final-run seeding enables deterministic cuDNN kernels and sets `CUBLAS_WORKSPACE_CONFIG`.
* The Optuna search deliberately uses a **time-based seed** and cuDNN autotuning, so the search itself is not bit-wise reproducible; the selected configurations are written explicitly in Section 3.1.
* Bootstrap / Dirichlet analyses use fixed seeds (2026, 20240601).
* Python 3.12.7 (kernel metadata). Main packages: `torch`, `torchvision`, `numpy`, `pandas`, `scikit-learn`, `scipy`, `matplotlib`, `seaborn`, `optuna`, `opencv-python`, `Pillow`, `openpyxl`, `tabulate`. Pin the exact versions with `pip freeze` for the supplementary material.

# 0 Setup and Shared Definitions

All definitions that are used by more than one section live here, so that every model, metric and dataset class is defined exactly once.

## 0.1 Imports

In [ ]:
import glob
import os
import random
import time
import warnings

# Required for deterministic cuBLAS kernels; must be set before CUDA is initialised.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import cv2
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from IPython.display import display
from PIL import Image
from scipy import stats
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.models import (
    AlexNet_Weights,
    DenseNet121_Weights,
    GoogLeNet_Weights,
    MobileNet_V2_Weights,
    ResNet18_Weights,
    VGG19_Weights,
)
from torchvision.utils import save_image

## 0.2 Global configuration

All file, folder, sheet and column names are in English. If you already have files from earlier runs under their former Indonesian names, convert them once with the cell in **Section 0.8** before running Section 1.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ---- Task definition -------------------------------------------------------------------
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]  # position = class index
LABEL_MAP = {name: idx for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# ---- Experimental protocol -------------------------------------------------------------
MODEL_NAMES = ["resnet18", "vgg19", "googlenet", "mobilenetv2", "densenet121", "alexnet"]
SPLIT_SEED = 42            # train/validation split, shuffling and augmentation
N_SEEDS = 30               # independent runs per backbone (Section 3)
MC_PASSES = 50             # Monte Carlo passes T used in the final evaluation
FOCUS_MODEL = "resnet18"   # backbone analysed in detail in Sections 3.4 - 3.9
FOCUS_SEED = 360           # run analysed in detail (one of the 30 seeds: 101 + 37 * 7)

# ---- Files and folders -----------------------------------------------------------------
TRAIN_CSV = "df_train.csv"     # original training images + offline augmentations
VAL_CSV = "df_validation.csv"    # validation images
TEST_CSV = "df_test.csv"       # official Testing folder
AUG_DIR = "augmented_train"   # folder with the augmented training images
SAVED_MODELS_DIR = "saved_models"
PLOTS_DIR = "plots"

## 0.3 Reproducibility helpers

In [ ]:
def set_seed(seed: int) -> None:
    """Fix every random number generator and force deterministic cuDNN kernels."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def set_dynamic_seed() -> int:
    """Time-based seed with cuDNN autotuning. Used only by the Optuna search (Section 2).

    The seed is returned so that it can be logged.
    """
    seed = int((time.time() * 100000) % 1000000)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
    return seed


def make_seeds(n_seeds: int = N_SEEDS) -> list:
    """Seed schedule of the final experiment: 101, 138, 175, ... (step 37)."""
    return [101 + i * 37 for i in range(n_seeds)]

## 0.4 Dataset class and evaluation transform

`BrainMRIDataset` reads a dataframe with the columns `path` and `label`. `convert_mode` is `"RGB"` everywhere except in the hyper-parameter search (Section 2), which loads images as grayscale and re-expands them to three channels.

In [ ]:
class BrainMRIDataset(Dataset):
    """Returns (image tensor, class index) for every row of a dataframe."""

    def __init__(self, df, transform=None, convert_mode="RGB"):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.convert_mode = convert_mode
        self.label_map = LABEL_MAP

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert(self.convert_mode)
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(self.label_map[row["label"]], dtype=torch.long)


# Standard evaluation transform (Section 3 onwards)
img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

## 0.5 MC Dropout layer and model factory

`MCDropout` keeps dropout active at inference time. Setting `enabled = False` turns it into an identity map, which gives the deterministic baseline of Sections 3.4 / 3.5 without rebuilding the network.

`get_model(name, dropout_rate, pretrained)` returns an ImageNet-pretrained backbone with a 4-class head. `pretrained=False` builds the same architecture with random weights; it is used when a fine-tuned `state_dict` is loaded afterwards (no ImageNet download needed).

In [ ]:
class MCDropout(nn.Module):
    """Dropout that stays active at inference time (Monte Carlo Dropout)."""

    def __init__(self, p: float = 0.5, enabled: bool = True):
        super().__init__()
        self.p = p
        self.enabled = enabled

    def forward(self, x):
        if not self.enabled:
            return x
        return F.dropout(x, p=self.p, training=True)


def get_model(name: str, dropout_rate: float = 0.5, pretrained: bool = True) -> nn.Module:
    """Backbone with a 4-class head in which every dropout layer is an MCDropout layer."""
    weights = (lambda w: w) if pretrained else (lambda w: None)

    if name == "resnet18":
        model = models.resnet18(weights=weights(ResNet18_Weights.IMAGENET1K_V1))
        model.fc = nn.Sequential(MCDropout(p=dropout_rate), nn.Linear(model.fc.in_features, NUM_CLASSES))

    elif name in ("vgg19", "alexnet"):
        if name == "vgg19":
            model = models.vgg19(weights=weights(VGG19_Weights.IMAGENET1K_V1))
        else:
            model = models.alexnet(weights=weights(AlexNet_Weights.IMAGENET1K_V1))
        classifier = list(model.classifier.children())
        for i, layer in enumerate(classifier):
            if isinstance(layer, nn.Dropout):
                classifier[i] = MCDropout(p=dropout_rate)
        classifier[-1] = nn.Linear(classifier[-1].in_features, NUM_CLASSES)
        model.classifier = nn.Sequential(*classifier)

    elif name == "googlenet":
        # transform_input=True keeps the forward pass identical to the ImageNet-pretrained model.
        model = models.googlenet(weights=weights(GoogLeNet_Weights.IMAGENET1K_V1),
                                 aux_logits=True, transform_input=True, init_weights=False)
        model.aux_logits = False
        model.aux1 = None
        model.aux2 = None
        model.dropout = MCDropout(p=dropout_rate)
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

    elif name == "mobilenetv2":
        model = models.mobilenet_v2(weights=weights(MobileNet_V2_Weights.IMAGENET1K_V1))
        model.classifier = nn.Sequential(MCDropout(p=dropout_rate),
                                         nn.Linear(model.classifier[1].in_features, NUM_CLASSES))

    elif name == "densenet121":
        model = models.densenet121(weights=weights(DenseNet121_Weights.IMAGENET1K_V1))
        model.classifier = nn.Sequential(MCDropout(p=dropout_rate),
                                         nn.Linear(model.classifier.in_features, NUM_CLASSES))
    else:
        raise ValueError(f"Model {name} is not supported")
    return model

## 0.6 Metric helpers

| Metric | Definition | Better |
|---|---|---|
| Accuracy, macro ROC-AUC (one-vs-rest), macro precision / recall / F1 | standard definitions on the MC-averaged probabilities | higher |
| Macro specificity | mean over classes of one-vs-rest $TN/(TN+FP)$ | higher |
| ECE | $\sum_{b=1}^{B} \frac{\lvert B_b\rvert}{N}\,\lvert \mathrm{acc}(B_b)-\mathrm{conf}(B_b)\rvert$ with $B=15$ equal-width confidence bins | lower |
| MCE | $\max_b \lvert \mathrm{acc}(B_b)-\mathrm{conf}(B_b)\rvert$ | lower |
| Brier score | $\frac{1}{N}\sum_i\sum_c (p_{ic}-y_{ic})^2$ | lower |
| NLL | $-\frac{1}{N}\sum_i \log p_{i,y_i}$ | lower |
| AURC | samples are sorted by decreasing confidence; the risk at coverage $k/N$ is the error rate among the $k$ most confident samples; AURC is the mean risk over all $k$ (area under the risk–coverage curve) | lower |

All metrics are computed on the **MC-averaged** probabilities $\bar p = \frac{1}{T}\sum_{t=1}^{T}\mathrm{softmax}(f(x;\text{mask}_t))$, and the prediction is $\arg\max_c \bar p_c$.

In [ ]:
def compute_calibration_metrics(probs, labels, n_bins=15):
    """ECE, MCE, Brier score and NLL.

    probs:  (N, C) mean predicted probabilities
    labels: (N,)   ground-truth class indices
    """
    preds = np.argmax(probs, axis=1)
    confidences = np.max(probs, axis=1)
    accuracies = (preds == labels).astype(float)

    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece, mce = 0.0, 0.0
    for i in range(n_bins):
        in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
        prop_in_bin = np.mean(in_bin)
        if prop_in_bin > 0:
            bin_error = np.abs(np.mean(accuracies[in_bin]) - np.mean(confidences[in_bin]))
            ece += bin_error * prop_in_bin
            mce = max(mce, bin_error)

    one_hot = np.eye(probs.shape[1])[labels]
    brier = np.mean(np.sum((probs - one_hot) ** 2, axis=1))
    nll = log_loss(labels, probs, labels=list(range(probs.shape[1])))
    return {"ece": ece, "mce": mce, "brier": brier, "nll": nll}


def compute_risk_coverage(probs, labels):
    """Area under the risk-coverage curve (AURC) for selective prediction; lower is better.

    Samples are sorted by DEcreasing confidence; the risk at coverage k/N is the error
    rate among the k most confident samples.
    """
    confidences = np.max(probs, axis=1)
    errors = (np.argmax(probs, axis=1) != labels).astype(float)

    order = np.argsort(-confidences, kind="mergesort")      # most confident first (stable)
    n = len(labels)
    coverages = np.arange(1, n + 1) / n                      # 1/n, 2/n, ..., 1
    risks = np.cumsum(errors[order]) / (n * coverages)       # error rate of the top-k samples
    return np.mean(risks)


def specificity_per_class(cm):
    """One-vs-rest specificity TN / (TN + FP) for each class (rows of `cm` = true class)."""
    total = cm.sum()
    specificity = np.zeros(cm.shape[0])
    for i in range(cm.shape[0]):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = total - tp - fp - fn
        specificity[i] = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return specificity


def compute_all_metrics(probs, labels):
    """Clinical, calibration and rejection metrics from (N, C) probabilities."""
    preds = np.argmax(probs, axis=1)
    try:
        roc_auc = roc_auc_score(np.eye(NUM_CLASSES)[labels], probs, average="macro", multi_class="ovr")
    except ValueError:
        roc_auc = 0.0

    class_ids = list(range(NUM_CLASSES))
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average=None, labels=class_ids, zero_division=0)
    cm = confusion_matrix(labels, preds, labels=class_ids)
    calibration = compute_calibration_metrics(probs, labels)

    return {
        "acc": np.mean(preds == labels),
        "roc_auc": roc_auc,
        "ece": calibration["ece"],
        "mce": calibration["mce"],
        "brier": calibration["brier"],
        "nll": calibration["nll"],
        "aurc": compute_risk_coverage(probs, labels),
        "macro_f1": np.mean(f1),
        "macro_precision": np.mean(precision),
        "macro_recall": np.mean(recall),
        "macro_specificity": np.mean(specificity_per_class(cm)),
    }

## 0.7 MC Dropout inference and uncertainty decomposition

For every input, $T$ stochastic forward passes are run with BatchNorm in evaluation mode. The predictive uncertainty is decomposed as

* **total** (predictive entropy): $H\big[\bar p\big]$
* **aleatoric** (expected entropy): $\mathbb{E}_t\,H[p_t]$
* **epistemic** (mutual information): $H[\bar p] - \mathbb{E}_t\,H[p_t]$

All three are expressed in nats.

In [ ]:
@torch.no_grad()
def mc_dropout_predict(model, dataloader, mc_passes=MC_PASSES):
    """T stochastic passes over a dataloader.

    Returns
    -------
    mc_probs : np.ndarray (N, T, C)  softmax probabilities of every pass
    labels   : np.ndarray (N,)
    """
    model.eval()  # BatchNorm frozen; MCDropout layers stay stochastic
    all_probs, all_labels = [], []
    for images, labels in dataloader:
        images = images.to(device)
        batch_probs = [torch.softmax(model(images), dim=1).cpu().numpy() for _ in range(mc_passes)]
        all_probs.append(np.stack(batch_probs, axis=1))      # (B, T, C)
        all_labels.append(labels.numpy())
    return np.vstack(all_probs), np.concatenate(all_labels)


def evaluate_mc_dropout(model, dataloader, mc_passes=MC_PASSES):
    """MC Dropout evaluation. Returns (metrics dict, mc_probs (N, T, C), labels (N,))."""
    mc_probs, labels = mc_dropout_predict(model, dataloader, mc_passes)
    return compute_all_metrics(mc_probs.mean(axis=1), labels), mc_probs, labels


def decompose_uncertainty(mc_probs, eps=1e-10):
    """Total / aleatoric / epistemic uncertainty from MC samples of shape (N, T, C)."""
    mean_probs = mc_probs.mean(axis=1)
    total = -np.sum(mean_probs * np.log(mean_probs + eps), axis=1)                 # H[E p]
    per_pass_entropy = -np.sum(mc_probs * np.log(mc_probs + eps), axis=2)          # H[p_t]
    aleatoric = per_pass_entropy.mean(axis=1)                                      # E H[p]
    return {"total": total, "aleatoric": aleatoric, "epistemic": total - aleatoric}

## 0.8 One-off conversion of files from earlier runs (optional)

Earlier versions of this notebook used Indonesian names for some files. The cell below converts such files **in place** and can be run repeatedly (it does nothing if there is nothing to convert):

* renames the files and the augmentation folder listed in `LEGACY_NAMES`;
* rewrites the image paths stored in `df_train.csv` (they contain the folder name);
* renames the sheets and columns inside the existing Excel workbooks (`LEGACY_SHEETS`, `LEGACY_COLUMNS`).

A target that already exists is never overwritten. Skip this cell if you start from scratch.

In [ ]:
LEGACY_NAMES = {   # former name -> current name
    "df_validasi.csv": VAL_CSV,
    "augmentasi_train": AUG_DIR,
    "hasil_180_runs_semua_model.xlsx": "results_180_runs_all_models.xlsx",
    "hasil_selective_prediction_30seeds.xlsx": "results_selective_prediction_30seeds.xlsx",
    "perbandingan_3_mode_resnet18.xlsx": "three_mode_comparison_resnet18.xlsx",
    **{f"hasil_30_seeds_{m}.xlsx": f"results_30_seeds_{m}.xlsx" for m in MODEL_NAMES},
}
LEGACY_SHEETS = {"ringkasan_statistik": "statistical_summary",
                 "auroc_selisih_skor": "auroc_score_differences"}
LEGACY_COLUMNS = {"perbandingan": "comparison",
                  "ci_tidak_memuat_0": "ci_excludes_zero",
                  "auroc_sd_antar_seed": "auroc_sd_across_seeds"}


def migrate_legacy_files():
    """Converts file names, stored image paths and Excel sheet/column names to the current English names."""
    changes = 0

    # 1) files and folders
    for old, new in LEGACY_NAMES.items():
        if not os.path.exists(old):
            continue
        if os.path.exists(new):
            print(f"[SKIP] both '{old}' and '{new}' exist - please resolve this manually.")
            continue
        os.rename(old, new)
        print(f"[RENAMED] {old} -> {new}")
        changes += 1

    # 2) image paths inside the training table
    if os.path.exists(TRAIN_CSV):
        df_train_paths = pd.read_csv(TRAIN_CSV)
        paths = df_train_paths["path"].astype(str)
        if paths.str.contains("augmentasi_train", regex=False).any():
            df_train_paths["path"] = paths.str.replace("augmentasi_train", AUG_DIR, regex=False)
            df_train_paths.to_csv(TRAIN_CSV, index=False)
            print(f"[UPDATED] image paths in {TRAIN_CSV}")
            changes += 1

    # 3) sheet and column names inside existing workbooks
    for path in (p for p in LEGACY_NAMES.values() if p.endswith(".xlsx") and os.path.exists(p)):
        sheets = pd.read_excel(path, sheet_name=None)
        needs_update = (any(name in LEGACY_SHEETS for name in sheets)
                        or any(col in LEGACY_COLUMNS for frame in sheets.values() for col in frame.columns))
        if not needs_update:
            continue
        with pd.ExcelWriter(path, engine="openpyxl") as writer:
            for name, frame in sheets.items():
                frame.rename(columns=LEGACY_COLUMNS).to_excel(
                    writer, sheet_name=LEGACY_SHEETS.get(name, name), index=False)
        print(f"[UPDATED] sheet / column names in {path}")
        changes += 1

    if changes == 0:
        print("No files from earlier runs found - nothing to convert.")


migrate_legacy_files()

# 1 Preprocessing Data

**Outputs of this section:** `df_train.csv` (original + augmented training images), `df_validation.csv`, `df_test.csv` (columns `id`, `label`, `original`, `path`) and the folder `augmented_train/` with the augmented images, plus printed split summaries.

## 1.1 Reading the brain tumor folders into a dataframe

Dataset: <https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset/data>. Expected layout: `Training/<class>/*.jpg` and `Testing/<class>/*.jpg`. File names are sorted so that the later stratified split does not depend on the order returned by the file system.

In [ ]:
SOURCE_DIRS = ["Training", "Testing"]

records = []
for source_dir in SOURCE_DIRS:
    if not os.path.exists(source_dir):
        print(f"Warning: folder '{source_dir}' not found.")
        continue
    for label in CLASS_NAMES:
        label_dir = os.path.join(source_dir, label)
        if not os.path.exists(label_dir):
            print(f"Warning: folder '{label_dir}' not found.")
            continue
        for filename in sorted(os.listdir(label_dir)):
            if filename.lower().endswith(".jpg"):
                records.append({
                    "id": os.path.splitext(filename)[0],      # file name without extension
                    "label": label,
                    "original": source_dir,                   # official folder of origin
                    "path": os.path.join(label_dir, filename),
                })

df_all = pd.DataFrame(records)
if df_all.empty:
    raise FileNotFoundError("No .jpg images found. Place 'Training/' and 'Testing/' next to this notebook.")

df_test = df_all[df_all["original"] == "Testing"].reset_index(drop=True)
df_train_raw = df_all[df_all["original"] == "Training"].reset_index(drop=True)

print("=== DATASET SCAN ===")
print(f"Images found    : {len(df_all)}")
print(f"  Training folder : {len(df_train_raw)}")
print(f"  Testing folder  : {len(df_test)}")
print("\n=== CLASS DISTRIBUTION PER OFFICIAL FOLDER ===")
print(pd.crosstab(df_all["label"], df_all["original"]))

## 1.2 Stratified train / validation split

The `Training` folder is split 80 % / 20 % (stratified by class, `random_state = 42`) into train and validation. The `Testing` folder is kept untouched as the test set.

In [ ]:
df_train_only, df_val = train_test_split(
    df_train_raw,
    test_size=0.20,
    random_state=SPLIT_SEED,
    stratify=df_train_raw["label"],
)
df_train_only = df_train_only.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

print("=== DATA SUMMARY ===")
print(f"Train (80 %)      : {len(df_train_only)}")
print(f"Validation (20 %) : {len(df_val)}")
print(f"Test              : {len(df_test)}")

split_view = pd.concat([
    df_train_only.assign(subset="train"),
    df_val.assign(subset="validation"),
    df_test.assign(subset="test"),
], ignore_index=True)
print("\n=== CLASS DISTRIBUTION PER SUBSET ===")
print(pd.crosstab(split_view["label"], split_view["subset"]))

## 1.3 Offline augmentation (training data only) and export

Each training image is augmented three times (horizontal flip, rotation ±15°, colour jitter, translation ±10 %, scale 0.9–1.1). Validation and test images are never augmented. The augmented copies are saved as PNG files and appended to the original training rows; the resulting table is shuffled and written to `df_train.csv`.

In [ ]:
AUG_PER_IMAGE = 3

aug_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=0),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
# Undo the normalisation so that the augmented images are saved as ordinary RGB images.
inv_normalize = transforms.Normalize(
    mean=[-m / s for m, s in zip(IMAGENET_MEAN, IMAGENET_STD)],
    std=[1 / s for s in IMAGENET_STD],
)

os.makedirs(AUG_DIR, exist_ok=True)
set_seed(SPLIT_SEED)  # makes the random augmentations repeatable

augmented_records = []
print("Starting augmentation of the training data ...")
for _, row in df_train_only.iterrows():
    try:
        image = Image.open(row["path"]).convert("RGB")
    except Exception as exc:
        print(f"Failed to open image {row['path']}: {exc}")
        continue

    for aug_idx in range(AUG_PER_IMAGE):
        augmented = inv_normalize(aug_transform(image))
        new_id = f"{row['id']}_aug{aug_idx:02d}"
        save_path = os.path.join(AUG_DIR, f"{new_id}_{row['label']}.png")
        save_image(augmented, save_path)
        augmented_records.append({
            "id": new_id,
            "label": row["label"],
            "original": row["original"],
            "path": save_path,
        })

aug_df = pd.DataFrame(augmented_records)
df_train_final = (pd.concat([df_train_only, aug_df], ignore_index=True)
                  .sample(frac=1, random_state=SPLIT_SEED)
                  .reset_index(drop=True))

df_train_final.to_csv(TRAIN_CSV, index=False)
df_val.to_csv(VAL_CSV, index=False)
df_test.to_csv(TEST_CSV, index=False)

print("\n=== FINAL DATASET SUMMARY ===")
print(f"{TRAIN_CSV:<16}: {len(df_train_final)} images (original {len(df_train_only)} + augmented {len(aug_df)})")
print(f"{VAL_CSV:<16}: {len(df_val)} images")
print(f"{TEST_CSV:<16}: {len(df_test)} images")
print(f"Columns: {list(df_train_final.columns)}")

## 1.4 Preview of the exported tables

In [ ]:
for name, frame in [("df_train_final", df_train_final), ("df_val", df_val), ("df_test", df_test)]:
    print(f"{name}: {frame.shape[0]} rows x {frame.shape[1]} columns")
    display(frame.head())

# 2 Hyper-parameter Optimisation of the Six Backbones (Optuna)

Each of the six backbones is tuned in its own Optuna study (25 trials, 150 trials in total).

| Hyper-parameter | Search space |
|---|---|
| MC dropout rate | 0.10 – 0.70, step 0.025 |
| Learning rate | 1e-6 – 1e-2 (log-uniform) |
| Weight decay | 1e-7 – 1e-2 (log-uniform) |
| Batch size | 16, 32, 64 |
| Optimiser | AdamW or SGD (momentum 0.9) |
| Epochs | 5 – 20 |

* **Objective:** maximise validation accuracy. Final validation accuracy uses the mean logits of 10 stochastic passes; the intermediate accuracy that feeds the pruner uses 2 passes.
* **Pruner:** `MedianPruner(n_startup_trials=5, n_warmup_steps=3)`.
* **Test accuracy** is logged for reporting only; it is never used to select a configuration.
* In this section images are loaded as grayscale and re-expanded to three channels (`Grayscale(3)`).

**Inputs:** `df_train.csv`, `df_validation.csv`, `df_test.csv`. **Outputs:** `df_report_<model>.csv` (one row per trial), `optuna_6models_combined_report.xlsx`.

## 2.1 Search datasets

In [ ]:
hpo_img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

hpo_train_dataset = BrainMRIDataset(pd.read_csv(TRAIN_CSV), transform=hpo_img_transform, convert_mode="L")
hpo_val_dataset = BrainMRIDataset(pd.read_csv(VAL_CSV), transform=hpo_img_transform, convert_mode="L")
hpo_test_dataset = BrainMRIDataset(pd.read_csv(TEST_CSV), transform=hpo_img_transform, convert_mode="L")
print(f"HPO datasets | train: {len(hpo_train_dataset)} | validation: {len(hpo_val_dataset)} | test: {len(hpo_test_dataset)}")

## 2.2 Objective function and study runner

In [ ]:
@torch.no_grad()
def eval_mean_logits(model, loader, passes):
    """Accuracy and probabilities from the mean logits of `passes` stochastic passes (HPO only)."""
    model.eval()
    correct, total = 0, 0
    probs, labels_all = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = torch.stack([model(images) for _ in range(passes)], dim=0).mean(dim=0)
        correct += logits.argmax(dim=1).eq(labels).sum().item()
        total += labels.size(0)
        probs.append(torch.softmax(logits, dim=1).cpu().numpy())
        labels_all.append(labels.cpu().numpy())
    return correct / total, np.vstack(probs), np.concatenate(labels_all)


def run_optuna_experiment(model_name: str, n_trials: int = 25) -> pd.DataFrame:
    """Runs one Optuna study for a single backbone and returns a per-trial report."""
    print(f"\n================ OPTIMISING MODEL: {model_name.upper()} ================")
    search_seed = set_dynamic_seed()
    print(f"Search seed (time-based): {search_seed}")

    def objective(trial):
        start_time = time.time()
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.7, step=0.025)
        lr = trial.suggest_float("lr", 1e-6, 1e-2, log=True)
        weight_decay = trial.suggest_float("weight_decay", 1e-7, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
        optimizer_name = trial.suggest_categorical("optimizer", ["adamW", "sgd"])
        n_epochs = trial.suggest_int("n_epochs", 5, 20)

        train_loader = DataLoader(hpo_train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
        val_loader = DataLoader(hpo_val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)
        test_loader = DataLoader(hpo_test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

        model = get_model(model_name, dropout_rate=dropout_rate).to(device)
        criterion = nn.CrossEntropyLoss()
        if optimizer_name == "adamW":
            optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        else:
            optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay, momentum=0.9)

        # ---- training loop with pruning on intermediate validation accuracy ----
        for epoch in range(n_epochs):
            model.train()
            correct_train, total_train = 0, 0
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                correct_train += outputs.argmax(dim=1).eq(labels).sum().item()
                total_train += labels.size(0)
            train_acc = correct_train / total_train

            intermediate_acc, _, _ = eval_mean_logits(model, val_loader, passes=2)
            trial.report(intermediate_acc, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        # ---- final evaluation (10 stochastic passes) ----
        val_accuracy, val_probs, val_labels = eval_mean_logits(model, val_loader, passes=10)
        try:
            val_roc_auc = roc_auc_score(np.eye(NUM_CLASSES)[val_labels], val_probs,
                                        average="macro", multi_class="ovr")
        except Exception:
            val_roc_auc = 0.0
        test_accuracy, _, _ = eval_mean_logits(model, test_loader, passes=10)

        trial.set_user_attr("train_acc", train_acc)
        trial.set_user_attr("val_roc_auc", val_roc_auc)
        trial.set_user_attr("test_acc", test_accuracy)
        trial.set_user_attr("epochs_run", n_epochs)
        trial.set_user_attr("duration_sec", time.time() - start_time)
        return val_accuracy

    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
    study = optuna.create_study(direction="maximize", pruner=pruner)
    study.optimize(objective, n_trials=n_trials)

    # ---- per-trial report (completed and pruned trials) ----
    rows = []
    for trial in study.trials:
        completed = trial.state == optuna.trial.TrialState.COMPLETE
        if completed:
            status = "COMPLETE"
        elif trial.state == optuna.trial.TrialState.PRUNED:
            status = "PRUNED"
        else:
            status = "FAILED"
        rows.append({
            "model_name": model_name,
            "trial_number": trial.number,
            "val_acc": trial.value if completed else np.nan,
            "test_acc": trial.user_attrs.get("test_acc", np.nan),
            "mc_dropout_rate": trial.params.get("dropout_rate"),
            "batch_size": trial.params.get("batch_size"),
            "lr": trial.params.get("lr"),
            "weight_decay": trial.params.get("weight_decay"),
            "optimizer": trial.params.get("optimizer"),
            "train_acc": trial.user_attrs.get("train_acc", np.nan),
            "val_roc_auc": trial.user_attrs.get("val_roc_auc", np.nan),
            "epochs_run": trial.user_attrs.get("epochs_run") if completed else len(trial.intermediate_values),
            "duration_sec": trial.user_attrs.get("duration_sec", np.nan),
            "status": status,
        })
    return pd.DataFrame(rows)

## 2.3 Running the optimisation for each model

A model whose `df_report_<model>.csv` already exists is skipped. Set `FORCE_RERUN_HPO = True` (or delete the file) to repeat a study.

In [ ]:
HPO_TRIALS = 25
FORCE_RERUN_HPO = False

for name in MODEL_NAMES:
    report_csv = f"df_report_{name}.csv"
    if os.path.exists(report_csv) and not FORCE_RERUN_HPO:
        print(f"'{report_csv}' already exists - skipping {name.upper()}.")
        continue
    df_report = run_optuna_experiment(name, n_trials=HPO_TRIALS)
    df_report.to_csv(report_csv, index=False)
    print(f"Saved: {report_csv}")

## 2.4 Summary of the hyper-parameter search

All per-model reports are merged, numbered with a `global_trial_id` and sorted by validation accuracy (pruned trials have no validation accuracy and appear last).

In [ ]:
print("Combining all model reports ...")
df_final_report = pd.concat(
    [pd.read_csv(f"df_report_{name}.csv") for name in MODEL_NAMES], ignore_index=True)
df_final_report.insert(0, "global_trial_id", range(1, len(df_final_report) + 1))
df_final_report = df_final_report.sort_values(by="val_acc", ascending=False).reset_index(drop=True)

output_name = "optuna_6models_combined_report.xlsx"
df_final_report.to_excel(output_name, index=False)

print(f"Saved {len(df_final_report)} trials to '{output_name}'.")
print(df_final_report["status"].value_counts().to_string())

Reference table: the three best completed trials of every backbone by validation accuracy. The configurations used in Section 3.1 are typed in explicitly (`BEST_CONFIGS`, rounded to three significant digits); use this table to cross-check them.

In [ ]:
completed = df_final_report[df_final_report["status"] == "COMPLETE"]
top3 = completed.groupby("model_name", sort=False).head(3)
display(top3[["model_name", "trial_number", "val_acc", "val_roc_auc", "mc_dropout_rate",
              "lr", "weight_decay", "batch_size", "optimizer", "epochs_run"]]
        .sort_values(["model_name", "val_acc"], ascending=[True, False])
        .reset_index(drop=True))

# 3 Final Experiment: 30 Runs per Backbone with the Selected Hyper-parameters

## 3.1 Clinical, calibration and rejection metrics

For every backbone the selected configuration is trained with **30 different seeds** (180 runs). Each run is evaluated with MC Dropout (T = 50) on the validation and the test set.

**CDSS composite score.** The 30-seed means of every model are min–max normalised *across models* and combined in three pillars:

* $S_{clinical}$ — mean of 6 normalised metrics (accuracy, ROC-AUC, macro F1, precision, recall, specificity; higher is better)
* $S_{calibration}$ — mean of 4 normalised metrics (ECE, MCE, Brier, NLL; lower is better)
* $S_{rejection}$ — normalised AURC (lower is better)

$$S_{total} = w_1 S_{clinical} + w_2 S_{calibration} + w_3 S_{rejection},\qquad (w_1,w_2,w_3) = (0.5,\,0.3,\,0.2)$$

**Inputs:** `df_train.csv`, `df_validation.csv`, `df_test.csv`. **Outputs:** `results_30_seeds_<model>.xlsx` (30 rows per model), `saved_models/<model>/<model>_seed_<seed>.pth`, `results_180_runs_all_models.xlsx` (8 sheets), the printed ranking table, the seed-normality analysis `accuracy_normality_analysis.xlsx` with `normality_plots/` (3.1.4), the Friedman / post-hoc results `results_advanced_statistical_tests.xlsx` (3.1.5), and the paired comparison `statistical_comparison_no_duration.xlsx` / `model_comparison_performance_only.png` (3.1.6).

### 3.1.1 Selected configurations and datasets

The values below were selected from the Optuna search of Section 2 (see the reference table in 2.4).

In [ ]:
BEST_CONFIGS = {
    "resnet18":    dict(dropout_rate=0.625, lr=1.10e-3, weight_decay=5.15e-5, batch_size=16, optimizer="sgd",   n_epochs=18),
    "vgg19":       dict(dropout_rate=0.275, lr=4.20e-5, weight_decay=9.02e-5, batch_size=16, optimizer="adamW", n_epochs=9),
    "googlenet":   dict(dropout_rate=0.575, lr=3.26e-4, weight_decay=5.32e-3, batch_size=64, optimizer="adamW", n_epochs=11),
    "mobilenetv2": dict(dropout_rate=0.475, lr=1.27e-4, weight_decay=3.37e-6, batch_size=64, optimizer="adamW", n_epochs=20),
    "densenet121": dict(dropout_rate=0.200, lr=3.28e-5, weight_decay=4.21e-4, batch_size=16, optimizer="adamW", n_epochs=7),
    "alexnet":     dict(dropout_rate=0.625, lr=1.19e-3, weight_decay=9.04e-5, batch_size=16, optimizer="sgd",   n_epochs=11),
}

# Datasets for Section 3 (RGB images, standard evaluation transform)
train_dataset = BrainMRIDataset(pd.read_csv(TRAIN_CSV), transform=img_transform)
val_dataset = BrainMRIDataset(pd.read_csv(VAL_CSV), transform=img_transform)
test_dataset = BrainMRIDataset(pd.read_csv(TEST_CSV), transform=img_transform)
print(f"Datasets | train: {len(train_dataset)} | validation: {len(val_dataset)} | test: {len(test_dataset)}")

### 3.1.2 Training and evaluation over 30 seeds

For every seed the model is trained from the ImageNet initialisation, evaluated with MC Dropout (T = 50) and its weights are stored in `saved_models/`. The function skips a backbone whose Excel file already exists.

In [ ]:
def run_single_model_30_seeds(model_config, n_seeds=N_SEEDS, mc_passes=MC_PASSES):
    """Trains / evaluates one backbone over `n_seeds` seeds and saves the results to Excel."""
    model_name = model_config["model_name"]
    output_excel = f"results_30_seeds_{model_name}.xlsx"

    if os.path.exists(output_excel):
        print(f"\nFile '{output_excel}' already exists. Skipping {model_name.upper()}.")
        return pd.read_excel(output_excel)

    os.makedirs(os.path.join(SAVED_MODELS_DIR, model_name), exist_ok=True)
    seeds = make_seeds(n_seeds)
    results = []
    print(f"\n================ STARTING {n_seeds} RUNS FOR MODEL: {model_name.upper()} (T={mc_passes}) ================")

    for seed_idx, seed in enumerate(seeds, 1):
        start_time = time.time()
        set_seed(seed)

        batch_size = model_config["batch_size"]
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

        model = get_model(model_name, dropout_rate=model_config["dropout_rate"]).to(device)
        criterion = nn.CrossEntropyLoss()
        if model_config["optimizer"] == "adamW":
            optimizer = optim.AdamW(model.parameters(), lr=model_config["lr"],
                                    weight_decay=model_config["weight_decay"])
        else:
            optimizer = optim.SGD(model.parameters(), lr=model_config["lr"],
                                  weight_decay=model_config["weight_decay"], momentum=0.9)

        # ---- training ----
        for epoch in range(model_config["n_epochs"]):
            model.train()
            correct_train, total_train = 0, 0
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                correct_train += outputs.argmax(dim=1).eq(labels).sum().item()
                total_train += labels.size(0)
        train_acc = correct_train / total_train  # accuracy of the last epoch

        # ---- checkpoint (needed by Sections 3.3 - 3.9) ----
        model_path = os.path.join(SAVED_MODELS_DIR, model_name, f"{model_name}_seed_{seed}.pth")
        torch.save(model.state_dict(), model_path)

        # ---- evaluation with T MC passes ----
        val_res, _, _ = evaluate_mc_dropout(model, val_loader, mc_passes=mc_passes)
        test_res, _, _ = evaluate_mc_dropout(model, test_loader, mc_passes=mc_passes)
        duration = time.time() - start_time

        print(f"[{seed_idx:2d}/{n_seeds}] Model: {model_name:<11} | Seed: {seed:<5} | "
              f"Train Acc: {train_acc:.4f} | Val Acc: {val_res['acc']:.4f} | "
              f"Test Acc: {test_res['acc']:.4f} | ECE: {test_res['ece']:.4f} | Time: {duration:.1f}s")

        results.append({
            "model_name": model_name,
            "seed": seed,
            "dropout_rate": model_config["dropout_rate"],
            "lr": model_config["lr"],
            "weight_decay": model_config["weight_decay"],
            "batch_size": batch_size,
            "optimizer": model_config["optimizer"],
            "n_epochs": model_config["n_epochs"],
            "train_acc": train_acc,
            "val_acc": val_res["acc"],
            "val_roc_auc": val_res["roc_auc"],
            "test_acc": test_res["acc"],
            "test_roc_auc": test_res["roc_auc"],
            "test_ece": test_res["ece"],
            "test_mce": test_res["mce"],
            "test_brier": test_res["brier"],
            "test_nll": test_res["nll"],
            "test_aurc": test_res["aurc"],
            "test_macro_f1": test_res["macro_f1"],
            "test_macro_precision": test_res["macro_precision"],
            "test_macro_recall": test_res["macro_recall"],
            "test_macro_specificity": test_res["macro_specificity"],
            "duration_sec": duration,
            "model_path": model_path,
        })

    df_result = pd.DataFrame(results)
    df_result.to_excel(output_excel, index=False)
    print(f"Saved {model_name.upper()} results to: {output_excel}")
    return df_result

In [ ]:
# Heavy step: 6 backbones x 30 seeds. Finished backbones are skipped, so the cell can be re-run.
for model_name, cfg in BEST_CONFIGS.items():
    run_single_model_30_seeds({"model_name": model_name, **cfg}, n_seeds=N_SEEDS)

### 3.1.3 Merging the 180 runs, summary statistics and composite ranking

`merge_and_summarize_results` writes one Excel workbook with eight sheets:

| Sheet | Content |
|---|---|
| `raw_180_runs` | every individual run |
| `statistical_summary` | mean, SD and a `mean ± SD` string for every metric |
| `mean_std_formatted` | display table (`mean ± SD`) |
| `mean_only`, `std_only`, `max_only`, `min_only` | mean / sample SD / maximum / minimum over the 30 seeds |
| `composite_score_ranking` | $S_{clinical}$, $S_{calibration}$, $S_{rejection}$, $S_{total}$ and the rank of every backbone |

The composite score is computed from the 4-decimal rounded means of sheet `mean_only`.

In [ ]:
CLINICAL_METRICS = ["test_acc", "test_roc_auc", "test_macro_f1",
                    "test_macro_precision", "test_macro_recall", "test_macro_specificity"]   # higher is better
CALIBRATION_METRICS = ["test_ece", "test_mce", "test_brier", "test_nll"]                    # lower is better
REJECTION_METRIC = "test_aurc"                                                              # lower is better
COMPOSITE_WEIGHTS = (0.5, 0.3, 0.2)   # (w1 clinical, w2 calibration, w3 rejection)

SUMMARY_METRICS = [
    "train_acc", "val_acc", "val_roc_auc",
    "test_acc", "test_roc_auc", "test_ece", "test_mce",
    "test_brier", "test_nll", "test_aurc", "test_macro_f1",
    "test_macro_precision", "test_macro_recall", "test_macro_specificity",
]


def min_max_normalize(values, higher_is_better=True):
    """Min-max scaling to [0, 1]; if all values are equal every score is 1."""
    values = np.asarray(values, dtype=float)
    lo, hi = values.min(), values.max()
    if hi == lo:
        return np.ones_like(values)
    scaled = (values - lo) / (hi - lo)
    return scaled if higher_is_better else 1.0 - scaled


def pillar_scores(means):
    """(M, 3) array [S_clinical, S_calibration, S_rejection]; `means` has one row per model."""
    s_clin = np.mean([min_max_normalize(means[c], True) for c in CLINICAL_METRICS], axis=0)
    s_cal = np.mean([min_max_normalize(means[c], False) for c in CALIBRATION_METRICS], axis=0)
    s_rej = min_max_normalize(means[REJECTION_METRIC], False)
    return np.column_stack([s_clin, s_cal, s_rej])


def merge_and_summarize_results(output_final="results_180_runs_all_models.xlsx",
                                w1=COMPOSITE_WEIGHTS[0], w2=COMPOSITE_WEIGHTS[1], w3=COMPOSITE_WEIGHTS[2]):
    """Merges all `results_30_seeds_*.xlsx` files into one 8-sheet workbook."""
    files = glob.glob("results_30_seeds_*.xlsx")
    if not files:
        print("No result files found yet.")
        return None
    df_merged = pd.concat([pd.read_excel(f) for f in files], ignore_index=True)

    summary_rows, formatted_rows, mean_rows, std_rows, max_rows, min_rows = [], [], [], [], [], []
    for model_name, group in df_merged.groupby("model_name"):
        base = {"model_name": model_name, "total_runs": len(group)}
        row, fmt_row, mean_row, std_row, max_row, min_row = (dict(base) for _ in range(6))
        for m in SUMMARY_METRICS:
            mean_val, std_val = group[m].mean(), group[m].std()      # sample SD (ddof = 1)
            formatted = f"{mean_val:.4f} ± {std_val:.4f}"
            row[f"{m}_mean"], row[f"{m}_std"], row[f"{m}_str"] = mean_val, std_val, formatted
            fmt_row[m] = formatted
            mean_row[m] = round(mean_val, 4)
            std_row[m] = round(std_val, 4)
            max_row[m] = round(group[m].max(), 4)
            min_row[m] = round(group[m].min(), 4)
        for rows, content in zip((summary_rows, formatted_rows, mean_rows, std_rows, max_rows, min_rows),
                                 (row, fmt_row, mean_row, std_row, max_row, min_row)):
            rows.append(content)

    df_summary, df_formatted = pd.DataFrame(summary_rows), pd.DataFrame(formatted_rows)
    df_mean, df_std = pd.DataFrame(mean_rows), pd.DataFrame(std_rows)
    df_max, df_min = pd.DataFrame(max_rows), pd.DataFrame(min_rows)

    # ---- composite score and ranking ----
    P = pillar_scores(df_mean)
    df_score = df_mean.copy()
    df_score["S_clinical"], df_score["S_calibration"], df_score["S_rejection"] = P[:, 0], P[:, 1], P[:, 2]
    df_score["Total_Score"] = (w1 * df_score["S_clinical"]
                               + w2 * df_score["S_calibration"]
                               + w3 * df_score["S_rejection"])
    score_cols = ["S_clinical", "S_calibration", "S_rejection", "Total_Score"]
    df_score[score_cols] = df_score[score_cols].round(4)
    df_score = df_score.sort_values(by="Total_Score", ascending=False).reset_index(drop=True)
    df_score.insert(1, "Rank", range(1, len(df_score) + 1))
    df_score = df_score[[
        "model_name", "Rank", "Total_Score", "S_clinical", "S_calibration", "S_rejection",
        "test_acc", "test_roc_auc", "test_macro_f1",
        "test_ece", "test_mce", "test_brier", "test_nll", "test_aurc",
    ]]

    with pd.ExcelWriter(output_final, engine="openpyxl") as writer:
        df_merged.to_excel(writer, sheet_name="raw_180_runs", index=False)
        df_summary.to_excel(writer, sheet_name="statistical_summary", index=False)
        df_formatted.to_excel(writer, sheet_name="mean_std_formatted", index=False)
        df_mean.to_excel(writer, sheet_name="mean_only", index=False)
        df_std.to_excel(writer, sheet_name="std_only", index=False)
        df_max.to_excel(writer, sheet_name="max_only", index=False)
        df_min.to_excel(writer, sheet_name="min_only", index=False)
        df_score.to_excel(writer, sheet_name="composite_score_ranking", index=False)

    print("\n================ CDSS COMPOSITE SCORE & MODEL RANKING ================")
    print(df_score[["Rank", "model_name", "Total_Score", "S_clinical", "S_calibration", "S_rejection"]]
          .to_string(index=False))
    print(f"\nAll 8 sheets saved to: {output_final}")
    return df_merged, df_score

In [ ]:
merge_output = merge_and_summarize_results()
if merge_output is not None:
    df_runs_all, df_composite = merge_output

### 3.1.4 Normality of the test accuracy over the 30 seeds

For every backbone the **Shapiro–Wilk test** is applied to its 30 test accuracies (one per seed). It checks whether parametric tests are justified: H0 = the accuracies come from a normal distribution; $p \ge 0.05$ means normality is not rejected, $p < 0.05$ means the distribution is not normal. The tests are exploratory (one per backbone, no multiplicity correction) and have limited power with $n = 30$, so every result is accompanied by a histogram with a kernel-density estimate and a Q–Q plot.

**Input:** `results_180_runs_all_models.xlsx` (sheet `raw_180_runs`; if it does not exist yet, the individual `results_30_seeds_*.xlsx` files are used). **Outputs:**

* `accuracy_normality_analysis.xlsx` – sheet `shapiro_summary_accuracy` (N, mean, SD, min, max, Shapiro–Wilk W, p-value, conclusion per model) and sheet `raw_30_seed_accuracy` (test accuracy, seeds × models),
* `normality_plots/accuracy_normality_<model>.png` – histogram + KDE and Q–Q plot of every model,
* `normality_plots/accuracy_boxplot_all_models.png` – box plot (mean marked) of all models,
* a printed summary table.

In [ ]:
def analyze_accuracy_normality(all_runs_xlsx="results_180_runs_all_models.xlsx",
                               output_xlsx="accuracy_normality_analysis.xlsx",
                               output_graph_folder="normality_plots"):
    """Shapiro-Wilk test of the test accuracy (test_acc) over the 30 seeds of every model.

    Writes a workbook (summary + raw accuracies), a histogram/KDE + Q-Q figure per model and one
    box plot of all models. Returns (summary dataframe, accuracies with seeds as rows and models as columns).
    """
    os.makedirs(output_graph_folder, exist_ok=True)

    # 1. read the data
    if os.path.exists(all_runs_xlsx):
        df_raw = pd.read_excel(all_runs_xlsx, sheet_name="raw_180_runs")
    else:
        files = glob.glob("results_30_seeds_*.xlsx")      # merged workbook not created yet
        if not files:
            print("Error: neither the merged workbook nor any 'results_30_seeds_*.xlsx' file was found.")
            return None
        df_raw = pd.concat([pd.read_excel(f) for f in files], ignore_index=True)

    # seeds as rows, models as columns
    df_pivot_acc = df_raw[["model_name", "seed", "test_acc"]].pivot(
        index="seed", columns="model_name", values="test_acc")

    shapiro_summary = []
    print("\n================ NORMALITY ANALYSIS OF TEST ACCURACY (30 SEEDS PER MODEL) ================")

    # The plot style is applied only inside this block so that later figures keep their own style.
    with sns.axes_style("whitegrid"), plt.rc_context({"font.size": 10}):
        for model_name in df_pivot_acc.columns:
            data_acc = df_pivot_acc[model_name].dropna().values
            acc = pd.Series(data_acc)
            n_samples = len(acc)

            if n_samples >= 3:
                stat, p_val = stats.shapiro(data_acc)
                is_normal = "Normal" if p_val >= 0.05 else "Not Normal"
                conclusion = ("Normally distributed (p >= 0.05)" if p_val >= 0.05
                              else "Not normally distributed / skewed (p < 0.05)")
            else:
                stat, p_val, is_normal, conclusion = np.nan, np.nan, "N/A", "Insufficient sample"

            shapiro_summary.append({
                "Model (BCNN)": model_name.upper(),
                "Number of Seeds (N)": n_samples,
                "Mean Accuracy": round(acc.mean(), 5),
                "Standard Deviation (SD)": round(acc.std(ddof=1), 5),    # sample SD
                "Min Accuracy": round(acc.min(), 5),
                "Max Accuracy": round(acc.max(), 5),
                "Shapiro-Wilk W": round(stat, 5),
                "p-value": p_val,
                "Distribution": is_normal,
                "Test Conclusion": conclusion,
            })
            if n_samples < 3:
                continue

            # ---- figure per model: histogram + KDE and Q-Q plot ----
            fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
            sns.histplot(data_acc, kde=True, ax=axes[0], color="skyblue", stat="density", bins=8)
            axes[0].set_title(f"Accuracy Distribution: {model_name.upper()}\n"
                              f"(Mean={acc.mean():.4f}, SD={acc.std(ddof=1):.4f})", fontsize=11, fontweight="bold")
            axes[0].set_xlabel("Test Accuracy")
            axes[0].set_ylabel("Density")

            stats.probplot(data_acc, dist="norm", plot=axes[1])
            axes[1].get_lines()[0].set_markerfacecolor("crimson")
            axes[1].get_lines()[0].set_markeredgecolor("darkred")
            axes[1].get_lines()[1].set_color("black")
            axes[1].get_lines()[1].set_linestyle("--")
            p_text = "p<0.0001" if p_val < 1e-4 else f"p={p_val:.4f}"
            axes[1].set_title(f"Q-Q Plot: {model_name.upper()}\n(Shapiro W={stat:.4f}, {p_text})",
                              fontsize=11, fontweight="bold")
            plt.tight_layout()
            plt.savefig(os.path.join(output_graph_folder, f"accuracy_normality_{model_name}.png"), dpi=300)
            plt.show()

        # ---- combined box plot of all models ----
        plt.figure(figsize=(10, 5))
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=FutureWarning)       # seaborn: palette without hue
            sns.boxplot(data=df_pivot_acc, palette="Set3", showmeans=True,
                        meanprops={"marker": "o", "markerfacecolor": "red", "markeredgecolor": "red"})
        plt.title("Comparison of Test Accuracy Distribution (30 Seeds) Across All BCNN Models",
                  fontsize=12, fontweight="bold")
        plt.ylabel("Test Accuracy")
        plt.xlabel("Model Architecture")
        plt.xticks(rotation=15)
        plt.tight_layout()
        plt.savefig(os.path.join(output_graph_folder, "accuracy_boxplot_all_models.png"), dpi=300)
        plt.show()

    df_summary = pd.DataFrame(shapiro_summary)
    with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
        df_summary.to_excel(writer, sheet_name="shapiro_summary_accuracy", index=False)
        df_pivot_acc.to_excel(writer, sheet_name="raw_30_seed_accuracy")

    print(f"\nDone. Workbook saved to: '{output_xlsx}'")
    print(f"Done. Distribution plots saved in folder: '{output_graph_folder}/'")
    print("\n--- SUMMARY OF THE ACCURACY NORMALITY TEST ---")
    print(df_summary[["Model (BCNN)", "Mean Accuracy", "p-value", "Distribution"]].to_string(
        index=False, formatters={"Mean Accuracy": "{:.5f}".format, "p-value": "{:.4g}".format}))
    return df_summary, df_pivot_acc

In [ ]:
normality_output = analyze_accuracy_normality()
if normality_output is not None:
    df_normality_summary, df_accuracy_by_seed = normality_output

### 3.1.5 Friedman test and post-hoc pairwise Wilcoxon tests (all six backbones)

Normality of the accuracies is not guaranteed (3.1.4), so the comparison of all six backbones uses non-parametric tests with the **seeds as blocks**:

1. **Friedman test (global).** H0: all backbones have the same distribution of test accuracy across the seeds. $p<0.05$ means that at least two backbones differ.
2. **Post-hoc: pairwise Wilcoxon signed-rank tests** for all $m = 15$ pairs of backbones, with the **Holm step-down correction**. With the raw p-values sorted ascending, $p_{(1)}\le\dots\le p_{(m)}$, the adjusted values are $p^{adj}_{(i)} = \min\!\big(1,\ \max_{j\le i}(m-j+1)\,p_{(j)}\big)$; a pair is significant if $p^{adj}<0.05$.

Only seeds for which every backbone has a value are used. The seed index is treated as a matching variable; see the note in Appendix B.

**Input:** the accuracy table of 3.1.4 (or, if it is not in memory, `accuracy_normality_analysis.xlsx`, sheet `raw_30_seed_accuracy`). **Outputs:** `results_advanced_statistical_tests.xlsx` (sheets `friedman_test`, `posthoc_wilcoxon_holm`, `raw_30_seed_accuracy`) and the printed results.

In [ ]:
def holm_adjust(p_values):
    """Holm step-down adjusted p-values, returned in the order of the input."""
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    order = np.argsort(p, kind="stable")
    stepped = (m - np.arange(m)) * p[order]                           # (m - j + 1) * p_(j)
    adjusted_sorted = np.minimum(np.maximum.accumulate(stepped), 1.0)  # monotone and capped at 1
    adjusted = np.empty(m)
    adjusted[order] = adjusted_sorted
    return adjusted


def run_friedman_and_posthoc(df_pivot=None,
                             file_excel="accuracy_normality_analysis.xlsx",
                             sheet_data="raw_30_seed_accuracy",
                             output_excel="results_advanced_statistical_tests.xlsx"):
    """Friedman test over all models (seeds = blocks) + pairwise Wilcoxon tests with Holm correction."""
    if df_pivot is None:
        if not os.path.exists(file_excel):
            print(f"File '{file_excel}' not found. Run Section 3.1.4 first.")
            return None
        df_pivot = pd.read_excel(file_excel, sheet_name=sheet_data, index_col="seed")

    df_pivot = df_pivot.dropna()            # complete blocks only (all models present for the seed)
    models = list(df_pivot.columns)

    print("\n================ 1. FRIEDMAN TEST (GLOBAL TEST) ================")
    print(f"Models tested  : {models}")
    print(f"Blocks (seeds) : {len(df_pivot)}")

    friedman_stat, friedman_p = stats.friedmanchisquare(*[df_pivot[m].values for m in models])
    print(f"Friedman Chi-Square Statistic : {friedman_stat:.5f}")
    print(f"p-value                       : {friedman_p:.5e}")
    if friedman_p < 0.05:
        print("Conclusion: there is a statistically SIGNIFICANT performance difference between at least two models.")
    else:
        print("Conclusion: no significant difference between the tested models.")

    df_friedman_res = pd.DataFrame([{
        "Test Name": "Friedman Test",
        "Chi-Square Statistic": round(friedman_stat, 5),
        "p-value": friedman_p,
        "Significant (p < 0.05)": "Yes" if friedman_p < 0.05 else "No",
        "Blocks (seeds)": len(df_pivot),
        "Models (k)": len(models),
    }])

    # ---- 2. post-hoc pairwise Wilcoxon signed-rank tests with Holm correction ----
    print("\n================ 2. POST-HOC PAIRWISE WILCOXON TEST (HOLM CORRECTION) ================")
    pairwise_results = []
    for i in range(len(models)):
        for j in range(i + 1, len(models)):
            m1, m2 = models[i], models[j]
            stat, p_val = stats.wilcoxon(df_pivot[m1], df_pivot[m2])
            pairwise_results.append({
                "Model A": m1,
                "Model B": m2,
                "Mean A": round(df_pivot[m1].mean(), 5),
                "Mean B": round(df_pivot[m2].mean(), 5),
                "Wilcoxon Stat": round(stat, 5),
                "p-value (Raw)": p_val,
            })

    df_pairwise = (pd.DataFrame(pairwise_results)
                   .sort_values(by="p-value (Raw)").reset_index(drop=True))
    df_pairwise["p-value (Holm-Adjusted)"] = holm_adjust(df_pairwise["p-value (Raw)"])
    df_pairwise["Significant (p_adj < 0.05)"] = np.where(
        df_pairwise["p-value (Holm-Adjusted)"] < 0.05, "Significant", "Not Significant")

    shown = df_pairwise[["Model A", "Model B", "p-value (Raw)", "p-value (Holm-Adjusted)",
                         "Significant (p_adj < 0.05)"]].copy()
    for col in ("p-value (Raw)", "p-value (Holm-Adjusted)"):
        shown[col] = shown[col].map("{:.3e}".format)
    print(shown.to_string(index=False))

    with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
        df_friedman_res.to_excel(writer, sheet_name="friedman_test", index=False)
        df_pairwise.to_excel(writer, sheet_name="posthoc_wilcoxon_holm", index=False)
        df_pivot.to_excel(writer, sheet_name="raw_30_seed_accuracy")

    print(f"\nDone. Results of the advanced statistical tests saved to: '{output_excel}'")
    return df_friedman_res, df_pairwise

In [ ]:
friedman_output = run_friedman_and_posthoc(
    df_accuracy_by_seed if "df_accuracy_by_seed" in globals() else None)
if friedman_output is not None:
    df_friedman_result, df_posthoc_result = friedman_output

### 3.1.6 Paired statistical comparison of two backbones (ResNet18 vs. DenseNet121)

Runs of the two backbones are paired by seed. For each metric the normality of the paired differences is checked with the Shapiro–Wilk test; a **paired t-test** is used if $p > 0.05$, otherwise the **Wilcoxon signed-rank test**. The result is shown as a table and as box plots. The significance column uses $p<0.05$ without correction for the four metrics tested.

**Inputs:** `results_30_seeds_resnet18.xlsx`, `results_30_seeds_densenet121.xlsx`. **Outputs:** `statistical_comparison_no_duration.xlsx`, `model_comparison_performance_only.png`.

In [ ]:
def paired_model_comparison(model_a="resnet18", model_b="densenet121",
                            labels=("ResNet18", "DenseNet121"),
                            metrics=("test_acc", "test_roc_auc", "test_macro_f1", "test_ece"),
                            colors=("#1f77b4", "#ff7f0e"),
                            output_xlsx="statistical_comparison_no_duration.xlsx",
                            output_png="model_comparison_performance_only.png"):
    try:
        df_a = pd.read_excel(f"results_30_seeds_{model_a}.xlsx")
        df_b = pd.read_excel(f"results_30_seeds_{model_b}.xlsx")
    except FileNotFoundError as exc:
        print(f"Error: file not found. {exc}")
        print("Make sure the Excel files of Section 3.1.2 are in the working directory.")
        return None

    # Pair the runs by seed
    df_pair = pd.merge(df_a, df_b, on="seed", suffixes=("_a", "_b"))

    print("Performing paired statistical significance tests ...")
    print("-" * 90)
    results = []
    for metric in metrics:
        col_a, col_b = f"{metric}_a", f"{metric}_b"
        diff = df_pair[col_a] - df_pair[col_b]

        _, shapiro_p = stats.shapiro(diff)               # normality of the paired differences
        if shapiro_p > 0.05:
            test_name = "Paired t-test"
            stat_val, p_value = stats.ttest_rel(df_pair[col_a], df_pair[col_b])
        else:
            test_name = "Wilcoxon Signed-Rank"
            stat_val, p_value = stats.wilcoxon(df_pair[col_a], df_pair[col_b])

        results.append({
            "Metric": metric,
            "Test Used": test_name,
            "Shapiro p-value": round(shapiro_p, 6),
            "Test Statistic": round(stat_val, 4),
            "p-value": f"{p_value:.2e}",
            f"Mean Diff ({labels[0]} - {labels[1]})": round(diff.mean(), 6),
            "Significant (p < 0.05)": "Yes" if p_value < 0.05 else "No",
        })

    df_results = pd.DataFrame(results)
    print(df_results.to_markdown(index=False))
    df_results.to_excel(output_xlsx, index=False)
    print(f"\nResults table saved to '{output_xlsx}'")

    # ---- box plots ----
    fig, axes = plt.subplots(1, len(metrics), figsize=(20, 5))
    for ax, metric in zip(np.atleast_1d(axes), metrics):
        data_plot = df_pair[[f"{metric}_a", f"{metric}_b"]].copy()
        data_plot.columns = list(labels)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=FutureWarning)   # seaborn: palette without hue
            sns.boxplot(data=data_plot, ax=ax, palette=list(colors), width=0.6)
        info = df_results[df_results["Metric"] == metric].iloc[0]
        ax.set_title(f"{metric}\np={info['p-value']} (significant: {info['Significant (p < 0.05)']})",
                     fontsize=11, fontweight="bold")
        ax.tick_params(axis="x", rotation=45)
        ax.grid(axis="y", alpha=0.3)

    plt.suptitle(f"Statistical Comparison: {labels[0]} vs {labels[1]} (Performance Only)", y=1.02, fontsize=14)
    plt.tight_layout()
    plt.savefig(output_png, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Box plot saved as '{output_png}'")
    return df_results

In [ ]:
df_pairwise_results = paired_model_comparison()

## 3.2 MCDA sensitivity analysis

*Reviewer-response reference: comment CL13 (No. 13).* The analysis re-examines the composite ranking of Section 3.1 **without retraining anything**.

| Part | Question | Method |
|---|---|---|
| A. Weight sensitivity | Does the ranking depend on the weights $(w_1,w_2,w_3)$? | 10,000 weight vectors drawn from Dirichlet(1,1,1); ranks are recomputed for each draw; share of draws in which each model takes every rank; Kendall $\tau$ against the baseline ranking (0.5 / 0.3 / 0.2); pairwise win probabilities |
| B. Weight scenarios | What happens under deliberately chosen weights? | eight fixed scenarios (equal weights, single pillar only, clinical-/calibration-/rejection-heavy) and a map of the top-ranked model over the whole weight simplex |
| C. Seed bootstrap | How much does the ranking depend on which seeds were run? | the 30 seeds are resampled with replacement (the same seeds for all models), 2,000 times, using the same min–max formulas; also combined with random Dirichlet weights |

**Important:** the `test_aurc` column must come from the corrected AURC (samples sorted by decreasing confidence, risk = error rate of the accepted samples). The routine warns if the mean AURC of a model exceeds its error rate.

**Input:** `results_180_runs_all_models.xlsx` (sheet `raw_180_runs`). **Outputs:** `mcda_sensitivity_results.xlsx` (9 sheets), `mcda_sensitivity.png`.

In [ ]:
MCDA_BASE_W = COMPOSITE_WEIGHTS
MCDA_METRICS = CLINICAL_METRICS + CALIBRATION_METRICS + [REJECTION_METRIC]
PRETTY_NAMES = {"resnet18": "ResNet18", "densenet121": "DenseNet121", "mobilenetv2": "MobileNetV2",
                "googlenet": "GoogLeNet", "vgg19": "VGG19", "alexnet": "AlexNet"}

MCDA_SCENARIOS = {
    "Baseline (0.5/0.3/0.2)": (0.5, 0.3, 0.2),
    "Equal weights (1/3 each)": (1 / 3, 1 / 3, 1 / 3),
    "Clinical only": (1, 0, 0),
    "Calibration only": (0, 1, 0),
    "Rejection only": (0, 0, 1),
    "Clinical-heavy (0.7/0.2/0.1)": (0.7, 0.2, 0.1),
    "Calibration-heavy (0.3/0.5/0.2)": (0.3, 0.5, 0.2),
    "Rejection-heavy (0.3/0.2/0.5)": (0.3, 0.2, 0.5),
}


def rank_matrix(S):
    """S: (N, M) scores -> ranks 1..M for each row (1 = best)."""
    order = np.argsort(-S, axis=1, kind="stable")
    R = np.empty_like(order)
    R[np.arange(S.shape[0])[:, None], order] = np.arange(1, S.shape[1] + 1)
    return R


def kendall_tau_vs(S, base):
    """Kendall tau-a of every row of S against the baseline score vector `base` (no ties)."""
    M = S.shape[1]
    total, n_pairs = 0.0, 0
    for i in range(M):
        for j in range(i + 1, M):
            total = total + np.sign(S[:, i] - S[:, j]) * np.sign(base[i] - base[j])
            n_pairs += 1
    return total / n_pairs


def rank_distribution(R, M):
    """(M, M) matrix: share of draws in which model m obtains rank r."""
    return np.array([[np.mean(R[:, m] == r) for r in range(1, M + 1)] for m in range(M)])


def dirichlet_analysis(P, names, n=10000, alpha=(1, 1, 1), seed=2026, base_w=MCDA_BASE_W):
    """Part A: ranking under random weights drawn from a Dirichlet distribution."""
    rng = np.random.default_rng(seed)
    W = rng.dirichlet(alpha, size=n)
    S = W @ P.T
    R = rank_matrix(S)
    base = P @ np.asarray(base_w)
    M = len(names)
    tau = kendall_tau_vs(S, base)
    pairwise = np.array([[np.mean(S[:, i] > S[:, j]) if i != j else np.nan for j in range(M)]
                         for i in range(M)])
    return dict(W=W, S=S, R=R, base=base, tau=tau,
                rank_dist=rank_distribution(R, M), pairwise=pairwise,
                same_ranking=float(np.mean(tau == 1.0)),
                same_top1=float(np.mean(R[:, int(np.argmax(base))] == 1)))


def simplex_grid(P, step=0.01):
    """Part B: top-ranked model for every weight vector on a regular grid of the simplex."""
    ws = [(a, b, 1 - a - b) for a in np.arange(0, 1 + 1e-9, step)
          for b in np.arange(0, 1 - a + 1e-9, step)]
    W = np.clip(np.array(ws), 0, 1)
    winners = np.argmax(W @ P.T, axis=1)
    return W, winners


def ranking_string(score, names):
    return " > ".join(names[i] for i in np.argsort(-score, kind="stable"))


def bootstrap_analysis(raw, models, n_boot=2000, seed=2026, base_w=MCDA_BASE_W):
    """Part C: resample the seeds (paired across models) and recompute the ranking."""
    common = sorted(set.intersection(*[set(raw.loc[raw.model_name == m, "seed"]) for m in models]))
    arr = np.stack([raw[raw.model_name == m].set_index("seed").loc[common, MCDA_METRICS].to_numpy(float)
                    for m in models])                       # (M, n_seed, K)
    n_seed, M = arr.shape[1], len(models)
    rng = np.random.default_rng(seed)
    bw = np.asarray(base_w)
    R_base, R_rand, S_tot, S_rej = [], [], [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n_seed, n_seed)               # the same seeds are used for all models
        means = pd.DataFrame(arr[:, idx, :].mean(axis=1), index=models, columns=MCDA_METRICS)
        P = pillar_scores(means)
        s = P @ bw
        S_tot.append(s)
        S_rej.append(P[:, 2])
        R_base.append(rank_matrix(s[None, :])[0])
        R_rand.append(rank_matrix((rng.dirichlet([1, 1, 1]) @ P.T)[None, :])[0])
    R_base, R_rand = np.array(R_base), np.array(R_rand)
    return dict(n_seed=n_seed,
                rank_dist_base=rank_distribution(R_base, M),
                rank_dist_joint=rank_distribution(R_rand, M),
                S_tot=np.array(S_tot), S_rej=np.array(S_rej))


def make_mcda_figure(res, names, grid_W, grid_win, base_w, path):
    """(a) rank distribution under random weights, (b) top-ranked model over the weight simplex."""
    M = len(names)
    colors = plt.get_cmap("tab10")(np.arange(M))
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={"width_ratios": [1.15, 1]})

    # (a) rank distribution
    rd = res["rank_dist"]
    shades = plt.get_cmap("Blues")(np.linspace(0.95, 0.15, M))
    left = np.zeros(M)
    for r in range(M):
        ax[0].barh(np.arange(M), rd[:, r], left=left, color=shades[r], edgecolor="white", label=f"Rank {r + 1}")
        for m in range(M):
            if rd[m, r] > 0.06:
                ax[0].text(left[m] + rd[m, r] / 2, m, f"{rd[m, r]:.0%}", ha="center", va="center",
                           fontsize=8, color="white" if r < 3 else "black")
        left += rd[:, r]
    ax[0].set_yticks(np.arange(M))
    ax[0].set_yticklabels(names)
    ax[0].invert_yaxis()
    ax[0].set_xlim(0, 1)
    ax[0].set_xlabel("Share of 10,000 Dirichlet weight draws")
    ax[0].set_title("(a) Rank distribution under random weights")
    ax[0].legend(ncol=M, fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.14), frameon=False)

    # (b) simplex map
    V = np.array([[0.5, np.sqrt(3) / 2], [0.0, 0.0], [1.0, 0.0]])   # clinical, calibration, rejection
    pts = grid_W @ V
    ax[1].scatter(pts[:, 0], pts[:, 1], c=colors[grid_win], s=6, marker="s", linewidths=0)
    ax[1].plot(*np.vstack([V, V[0]]).T, color="k", lw=0.8)
    bp = np.asarray(base_w) @ V
    ax[1].scatter(*bp, marker="*", s=160, c="white", edgecolors="k", zorder=5)
    ax[1].annotate("baseline", bp, xytext=(8, 6), textcoords="offset points", fontsize=8)
    for v, text, offset in zip(V, ["Clinical (w1=1)", "Calibration (w2=1)", "Rejection (w3=1)"],
                               [(0, 6), (-4, -14), (4, -14)]):
        ax[1].annotate(text, v, xytext=offset, textcoords="offset points", ha="center", fontsize=8)
    handles = [plt.Line2D([], [], marker="s", ls="", color=colors[i], label=names[i]) for i in np.unique(grid_win)]
    ax[1].legend(handles=handles, fontsize=8, loc="upper right", frameon=False)
    ax[1].set_aspect("equal")
    ax[1].axis("off")
    ax[1].set_title("(b) Top-ranked model over the weight simplex")
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.show()
    plt.close(fig)


def run_mcda_sensitivity(xlsx_path="results_180_runs_all_models.xlsx", sheet="raw_180_runs",
                         n_draws=10000, n_boot=2000, seed=2026, out_prefix="mcda_sensitivity"):
    raw = pd.read_excel(xlsx_path, sheet_name=sheet)
    raw["model_name"] = raw["model_name"].str.lower()
    means = raw.groupby("model_name")[MCDA_METRICS].mean()

    # Sanity check of the AURC: it must not exceed the error rate if confidence is informative
    for m, r in means.iterrows():
        err = 1 - r["test_acc"]
        if r["test_aurc"] > err:
            print(f"[WARNING] {m}: AURC {r['test_aurc']:.4f} > error rate {err:.4f}. "
                  f"Check the AURC implementation before using these results.")

    # ---- pillar scores and baseline ranking ----
    P = pillar_scores(means)
    order = np.argsort(-(P @ np.asarray(MCDA_BASE_W)), kind="stable")
    models = [means.index[i] for i in order]
    means, P = means.loc[models], P[order]
    names = [PRETTY_NAMES.get(m, m) for m in models]
    base = P @ np.asarray(MCDA_BASE_W)
    base_tbl = pd.DataFrame(P, index=names, columns=["S_clinical", "S_calibration", "S_rejection"])
    base_tbl["S_total"] = base
    base_tbl.insert(0, "Rank", range(1, len(names) + 1))
    print("\n=== Baseline (w = 0.5/0.3/0.2) ===")
    print(base_tbl.round(4).to_string())

    # ---- consistency with sheet composite_score_ranking (if present) ----
    try:
        comp = pd.read_excel(xlsx_path, sheet_name="composite_score_ranking").set_index("model_name")
        diff = (comp["Total_Score"] - pd.Series(base, index=models)).abs().max()
        print(f"\nMax. difference vs sheet composite_score_ranking: {diff:.4f}"
              + ("  <-- MISMATCH, please check" if diff > 5e-4 else "  (match)"))
    except Exception:
        pass

    # ---- A. Dirichlet weights ----
    D = dirichlet_analysis(P, names, n=n_draws, seed=seed)
    rank_cols = [f"P(rank={r})" for r in range(1, len(names) + 1)]
    rd = pd.DataFrame(D["rank_dist"], index=names, columns=rank_cols)
    rd["mean_rank"] = (D["rank_dist"] * np.arange(1, len(names) + 1)).sum(1)
    print(f"\n=== A. Dirichlet(1,1,1), n = {n_draws} ===")
    print(rd.round(3).to_string())
    tau = D["tau"]
    print(f"Kendall tau vs baseline: mean {tau.mean():.3f}, median {np.median(tau):.3f}, "
          f"P5 {np.percentile(tau, 5):.3f}, min {tau.min():.3f}")
    print(f"Ranking identical to baseline: {D['same_ranking']:.1%} | "
          f"baseline top-1 model stays #1: {D['same_top1']:.1%}")
    pair = pd.DataFrame(D["pairwise"], index=names, columns=names)

    # ---- B. Fixed scenarios and simplex map ----
    rows = []
    for scenario, w in MCDA_SCENARIOS.items():
        s = P @ np.asarray(w)
        rows.append({"Scenario": scenario, "w1": round(w[0], 3), "w2": round(w[1], 3), "w3": round(w[2], 3),
                     "Top-1": names[int(np.argmax(s))], "Ranking": ranking_string(s, names)})
    scen = pd.DataFrame(rows)
    print("\n=== B. Weight scenarios ===")
    print(scen.to_string(index=False))
    grid_W, grid_win = simplex_grid(P)
    area = pd.Series([np.mean(grid_win == i) for i in range(len(names))], index=names, name="Share of simplex")
    print("\nShare of the weight simplex in which each model ranks first:")
    print(area.round(3).to_string())

    # ---- C. Seed bootstrap ----
    B = bootstrap_analysis(raw, models, n_boot=n_boot, seed=seed)
    pb = pd.DataFrame(B["rank_dist_base"], index=names, columns=rank_cols)
    pj = pd.DataFrame(B["rank_dist_joint"], index=names, columns=rank_cols)
    ci = pd.DataFrame({"S_total mean": B["S_tot"].mean(0),
                       "S_total 2.5%": np.percentile(B["S_tot"], 2.5, axis=0),
                       "S_total 97.5%": np.percentile(B["S_tot"], 97.5, axis=0),
                       "S_rejection 2.5%": np.percentile(B["S_rej"], 2.5, axis=0),
                       "S_rejection 97.5%": np.percentile(B["S_rej"], 97.5, axis=0)}, index=names)
    print(f"\n=== C. Seed bootstrap: {n_boot} resamples over {B['n_seed']} seeds (baseline weights) ===")
    print(pb.round(3).to_string())
    print("\nS_total and S_rejection (95% interval):")
    print(ci.round(3).to_string())
    print("\nBootstrap + random Dirichlet weights (joint):")
    print(pj.round(3).to_string())

    # ---- save ----
    with pd.ExcelWriter(out_prefix + "_results.xlsx", engine="openpyxl") as writer:
        base_tbl.round(4).to_excel(writer, sheet_name="baseline")
        rd.round(4).to_excel(writer, sheet_name="dirichlet_rank_dist")
        (pd.DataFrame({"tau": tau}).describe(percentiles=[.05, .25, .5, .75, .95]).round(4)
         .to_excel(writer, sheet_name="dirichlet_kendall_tau"))
        pair.round(4).to_excel(writer, sheet_name="dirichlet_pairwise")
        scen.to_excel(writer, sheet_name="weight_scenarios", index=False)
        area.round(4).to_frame().to_excel(writer, sheet_name="simplex_area")
        pb.round(4).to_excel(writer, sheet_name="bootstrap_baseline_w")
        ci.round(4).to_excel(writer, sheet_name="bootstrap_ci")
        pj.round(4).to_excel(writer, sheet_name="bootstrap_joint")
    make_mcda_figure(D, names, grid_W, grid_win, MCDA_BASE_W, out_prefix + ".png")
    print(f"\nSaved: {out_prefix}_results.xlsx and {out_prefix}.png")

In [ ]:
run_mcda_sensitivity()

## 3.3 Selective prediction over 30 seeds and error-detection AUROC

*Reviewer-response references: No. 15 · CL18 (selective prediction) and No. 21 · CL14 (error-detection AUROC with bootstrap CI).*

**Pipeline**

1. **Stage 1 – score extraction.** The checkpoint of every (model, seed) is loaded, MC Dropout (T = 50) is run on validation and test, and three uncertainty scores are stored in a compressed cache file `uncertainty_cache/<model>_seed_<seed>.npz` (the stage can be resumed):
   * `mi` – mutual information (epistemic),
   * `entropy` – predictive entropy (total),
   * `one_minus_maxsoftmax` – $1-\max_c \bar p_c$.
   Larger score = less certain.
2. **Stage 2 – selective prediction (per seed).** A prediction is *accepted* if its score is $\le \tau$. The threshold $\tau$ is **calibrated on the validation set and then applied unchanged to the test set**, under two policies:
   * *target coverage* (95 %, 90 %, 80 %): $\tau$ = the corresponding quantile of the validation scores;
   * *target selective accuracy* (99 %): the largest validation coverage whose selective accuracy is $\ge$ 99 %.
   Reported per seed and as mean ± SD across seeds: coverage, selective accuracy, number of errors rejected, error-rejection rate (share of all errors that were rejected) and rejected-error rate (share of rejected samples that were actually errors).
3. **Stage 3 – error-detection AUROC.** For every score, the AUROC for separating wrong (positive) from correct predictions on the test set. The 95 % confidence interval comes from a **bootstrap over test images** (2,000 resamples; the same resampled images are used for all seeds, models and scores, so differences between scores are paired). Differences between scores are reported with their own bootstrap CI.

**Inputs:** `df_validation.csv`, `df_test.csv`, `results_30_seeds_<model>.xlsx` (needs the `model_path` column) and the checkpoints in `saved_models/`. **Outputs:** `uncertainty_cache/*.npz`, `results_selective_prediction_30seeds.xlsx` (6 sheets: `selective_per_seed`, `selective_mean_sd`, `selective_numeric`, `auroc_per_seed`, `auroc_ci_bootstrap`, `auroc_score_differences`) and two printed tables.

### 3.3.1 Definitions

In [ ]:
INFER_BATCH = 64
NUM_WORKERS = 0
SCORE_NAMES = ["mi", "entropy", "one_minus_maxsoftmax"]   # larger = more uncertain
TARGET_COVERAGES = [0.95, 0.90, 0.80]    # policy A: threshold = quantile of the validation scores
TARGET_SEL_ACCS = [0.99]                 # policy B: largest validation coverage with selective accuracy >= target
PRIMARY_COVERAGE = 0.90                  # only used for the compact console summary
N_BOOT = 2000
BOOT_SEED = 20240601
CACHE_DIR = "uncertainty_cache"
OUT_XLSX = "results_selective_prediction_30seeds.xlsx"
EPS = 1e-12


# ---- uncertainty scores -----------------------------------------------------------------
def uncertainty_from_sums(sum_p, sum_h, T):
    """Scores from running sums over T MC passes.

    sum_p: (N, C) sum of the softmax outputs;  sum_h: (N,) sum of the per-pass entropies.
      entropy              : H[ mean_t p_t ]                  (total)
      mi                   : H[ mean_t p_t ] - mean_t H[p_t]   (epistemic)
      one_minus_maxsoftmax : 1 - max_c mean_t p_{t,c}
    """
    mean_p = sum_p / T
    mean_h = sum_h / T
    h_mean = -(mean_p * np.log(np.clip(mean_p, EPS, None))).sum(axis=1)
    mi = np.clip(h_mean - mean_h, 0.0, None)            # remove tiny negative values (round-off)
    scores = {"mi": mi, "entropy": h_mean, "one_minus_maxsoftmax": 1.0 - mean_p.max(axis=1)}
    return scores, mean_p


@torch.no_grad()
def mc_dropout_uncertainty(model, loader, mc_passes=MC_PASSES):
    """MC Dropout with BatchNorm frozen and MCDropout active.

    Accumulation is done in float64 so that the softmax does not saturate to exactly 1.0
    (which would create many tied scores).
    """
    model.eval()
    P, H, Y = [], [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        sum_p = torch.zeros(images.size(0), NUM_CLASSES, dtype=torch.float64, device=device)
        sum_h = torch.zeros(images.size(0), dtype=torch.float64, device=device)
        for _ in range(mc_passes):
            p = torch.softmax(model(images).double(), dim=1)
            sum_p += p
            sum_h += -(p * torch.log(p.clamp_min(EPS))).sum(dim=1)
        P.append(sum_p.cpu().numpy())
        H.append(sum_h.cpu().numpy())
        Y.append(labels.numpy())
    scores, mean_p = uncertainty_from_sums(np.vstack(P), np.concatenate(H), mc_passes)
    return {"labels": np.concatenate(Y), "mean_probs": mean_p, **scores}


# ---- Stage 1: cached score extraction ---------------------------------------------------
def read_model_runs(model_name):
    """seed, dropout rate and checkpoint path of every run in results_30_seeds_<model>.xlsx."""
    path = f"results_30_seeds_{model_name}.xlsx"
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found. Run the 30-seed experiment (Section 3.1) for {model_name} first.")
    df = pd.read_excel(path)
    runs = []
    for _, r in df.iterrows():
        seed = int(r["seed"])
        model_path = str(r["model_path"]) if "model_path" in df.columns else ""
        if not os.path.exists(model_path):
            model_path = os.path.join(SAVED_MODELS_DIR, model_name, f"{model_name}_seed_{seed}.pth")
        runs.append({"seed": seed, "dropout_rate": float(r["dropout_rate"]), "model_path": model_path})
    return runs


def cache_path(model_name, seed):
    return os.path.join(CACHE_DIR, f"{model_name}_seed_{seed}.npz")


def extract_all_scores(models_to_run=MODEL_NAMES):
    """Stage 1: store MC Dropout uncertainty scores of every (model, seed) on validation and test."""
    os.makedirs(CACHE_DIR, exist_ok=True)

    def make_loader(csv_path):
        return DataLoader(BrainMRIDataset(pd.read_csv(csv_path), transform=img_transform),
                          batch_size=INFER_BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    val_loader, test_loader = make_loader(VAL_CSV), make_loader(TEST_CSV)

    for model_name in models_to_run:
        runs = read_model_runs(model_name)
        print(f"\n===== {model_name.upper()}: {len(runs)} seeds, T={MC_PASSES} =====")
        for i, run in enumerate(runs, 1):
            seed, cp = run["seed"], cache_path(model_name, run["seed"])
            if os.path.exists(cp):
                print(f"[{i:2d}/{len(runs)}] seed {seed}: cache found, skipped")
                continue
            t0 = time.time()
            set_seed(seed)  # fixes the MC dropout masks so that this inference can be repeated
            model = get_model(model_name, dropout_rate=run["dropout_rate"], pretrained=False).to(device)
            model.load_state_dict(torch.load(run["model_path"], map_location=device, weights_only=True))
            val = mc_dropout_uncertainty(model, val_loader)
            test = mc_dropout_uncertainty(model, test_loader)
            np.savez_compressed(cp,
                                **{f"val_{k}": v for k, v in val.items()},
                                **{f"test_{k}": v for k, v in test.items()})
            del model
            if device.type == "cuda":
                torch.cuda.empty_cache()
            print(f"[{i:2d}/{len(runs)}] seed {seed}: val acc {np.mean(val['mean_probs'].argmax(1) == val['labels']):.4f} | "
                  f"test acc {np.mean(test['mean_probs'].argmax(1) == test['labels']):.4f} | {time.time() - t0:.1f}s")


def load_store(models_to_run=MODEL_NAMES):
    """Loads all caches -> store[(model, seed)] = dict, seeds_by_model[model] = [seed, ...]."""
    store, seeds_by_model, ref_labels = {}, {}, None
    for m in models_to_run:
        seeds_by_model[m] = []
        for run in read_model_runs(m):
            cp = cache_path(m, run["seed"])
            if not os.path.exists(cp):
                raise FileNotFoundError(f"Cache {cp} does not exist yet. Run extract_all_scores() first.")
            d = dict(np.load(cp))
            if ref_labels is None:
                ref_labels = d["test_labels"]
            # all seeds and models must use the same test images in the same order
            assert np.array_equal(ref_labels, d["test_labels"]), f"Test order/labels differ: {m} seed {run['seed']}"
            store[(m, run["seed"])] = d
            seeds_by_model[m].append(run["seed"])
    return store, seeds_by_model


# ---- Stage 2: selective prediction (calibrate on validation -> apply to test) -----------
# Acceptance rule: accept a prediction if score <= tau (a score equal to tau is accepted).
def threshold_for_coverage(val_scores, target_cov):
    """Policy A: tau = quantile of the validation scores so that the validation coverage >= target."""
    s = np.sort(val_scores)
    k = int(np.ceil(target_cov * len(s)))
    return s[max(k, 1) - 1]


def threshold_for_sel_acc(val_scores, val_correct, target_acc):
    """Policy B: largest validation coverage whose selective accuracy >= target.

    Returns NaN (nothing is accepted) if no threshold satisfies the target.
    """
    order = np.argsort(val_scores, kind="stable")
    s = val_scores[order]
    c = val_correct[order].astype(float)
    cum_acc = np.cumsum(c) / np.arange(1, len(s) + 1)
    last_in_tie = np.r_[s[1:] != s[:-1], True]          # evaluate only at the end of a group of tied scores
    ok = np.where(last_in_tie & (cum_acc >= target_acc))[0]
    return s[ok[-1]] if len(ok) else np.nan


def coverage_and_acc(scores, correct, tau):
    accept = scores <= tau                                # tau = NaN -> nothing accepted
    n_acc = int(accept.sum())
    return n_acc / len(scores), (correct[accept].mean() if n_acc > 0 else np.nan)


def apply_selective(scores, correct, tau):
    accept = scores <= tau
    errors = ~correct
    n, n_acc = len(scores), int(accept.sum())
    n_rej, n_err = n - n_acc, int(errors.sum())
    err_rej = int((errors & ~accept).sum())
    return {
        "coverage": n_acc / n,
        "selective_acc": correct[accept].mean() if n_acc > 0 else np.nan,
        "n_rejected": n_rej,
        "n_errors_total": n_err,
        "errors_rejected": err_rej,                                           # number of errors that were rejected
        "error_rejection_rate": err_rej / n_err if n_err > 0 else np.nan,     # share of ALL errors that were rejected
        "rejected_error_rate": err_rej / n_rej if n_rej > 0 else np.nan,      # share of rejected samples that were errors
    }


def selective_rows_for_seed(model_name, seed, d):
    val_correct = d["val_mean_probs"].argmax(1) == d["val_labels"]
    test_correct = d["test_mean_probs"].argmax(1) == d["test_labels"]
    base = {"model_name": model_name, "seed": seed,
            "val_acc": val_correct.mean(), "test_acc": test_correct.mean()}
    rows = []
    for s in SCORE_NAMES:
        v, t = d[f"val_{s}"], d[f"test_{s}"]
        policies = [("target_coverage", tc, threshold_for_coverage(v, tc)) for tc in TARGET_COVERAGES]
        policies += [("target_sel_acc", ta, threshold_for_sel_acc(v, val_correct, ta)) for ta in TARGET_SEL_ACCS]
        for policy, target, tau in policies:
            val_cov, val_sel = coverage_and_acc(v, val_correct, tau)
            rows.append({**base, "score": s, "policy": policy, "target": target, "threshold": tau,
                         "val_coverage": val_cov, "val_sel_acc": val_sel,
                         **apply_selective(t, test_correct, tau)})
    return rows


SEL_METRICS = ["coverage", "selective_acc", "errors_rejected", "error_rejection_rate",
               "rejected_error_rate", "n_errors_total"]


def summarize_selective(df):
    keys = ["model_name", "policy", "target", "score"]
    g = df.groupby(keys, sort=False)
    num = g[SEL_METRICS].agg(["mean", "std"])             # std = sample SD (ddof = 1); NaN ignored
    num.columns = [f"{m}_{stat}" for m, stat in num.columns]
    num["n_seeds"] = g.size()
    num["n_seeds_with_accepted"] = g["selective_acc"].count()
    num = num.reset_index()

    fmt = num[keys + ["n_seeds", "n_seeds_with_accepted"]].copy()
    for m in SEL_METRICS:
        digits = 2 if m in ("errors_rejected", "n_errors_total") else 4
        fmt[m] = [f"{a:.{digits}f} ± {b:.{digits}f}" for a, b in zip(num[f"{m}_mean"], num[f"{m}_std"])]
    return fmt, num


# ---- Stage 3: error-detection AUROC with bootstrap over test images --------------------
def make_bootstrap_counts(n_images, n_boot=N_BOOT, seed=BOOT_SEED):
    """Multiplicity matrix M (n_boot, n_images): M[b, i] = how often image i is drawn in resample b.

    The same M is used for all seeds / models / scores, which makes the bootstrap paired.
    """
    rng = np.random.default_rng(seed)
    return rng.multinomial(n_images, np.full(n_images, 1.0 / n_images), size=n_boot)


def bootstrap_auroc(scores, correct, M):
    """AUROC (positive = misclassified, high score = wrong prediction) for every resample.

    Exact and vectorised: AUROC_b = sum_ij m_bi m_bj W_ij / (sum_i m_bi * sum_j m_bj), with
    W_ij = 1[s_i > s_j] + 0.5 * 1[s_i = s_j] (i = wrong image, j = correct image; ties count 0.5).
    """
    err = ~correct
    e_idx, c_idx = np.where(err)[0], np.where(~err)[0]
    if len(e_idx) == 0 or len(c_idx) == 0:
        return np.full(M.shape[0], np.nan)
    se, sc = scores[e_idx][:, None], scores[c_idx][None, :]
    W = (se > sc).astype(np.float64) + 0.5 * (se == sc)
    Me, Mc = M[:, e_idx].astype(np.float64), M[:, c_idx].astype(np.float64)
    num = ((Me @ W) * Mc).sum(axis=1)
    den = Me.sum(axis=1) * Mc.sum(axis=1)
    return np.where(den > 0, num / np.where(den > 0, den, 1.0), np.nan)


def auroc_point(scores, correct):
    err = (~correct).astype(int)
    return roc_auc_score(err, scores) if 0 < err.sum() < len(err) else np.nan


DIFF_PAIRS = [("entropy", "mi"), ("one_minus_maxsoftmax", "mi"), ("one_minus_maxsoftmax", "entropy")]


def auroc_analysis(store, seeds_by_model, n_boot=N_BOOT):
    n_images = len(next(iter(store.values()))["test_labels"])
    M = make_bootstrap_counts(n_images, n_boot)

    seed_rows, summary_rows, diff_rows = [], [], []
    for m, seeds in seeds_by_model.items():
        point = {s: [] for s in SCORE_NAMES}              # per-seed AUROC
        boot = {s: [] for s in SCORE_NAMES}               # per-seed bootstrap distribution (n_boot,)
        for seed in seeds:
            d = store[(m, seed)]
            correct = d["test_mean_probs"].argmax(1) == d["test_labels"]
            for s in SCORE_NAMES:
                a = auroc_point(d[f"test_{s}"], correct)
                point[s].append(a)
                boot[s].append(bootstrap_auroc(d[f"test_{s}"], correct, M))
                seed_rows.append({"model_name": m, "seed": seed, "score": s, "auroc_error_detection": a})

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)    # seeds without errors -> NaN
            seedmean_boot = {s: np.nanmean(np.stack(boot[s], axis=1), axis=1) for s in SCORE_NAMES}
            for s in SCORE_NAMES:
                pt = np.array(point[s], dtype=float)
                lo, hi = np.nanpercentile(seedmean_boot[s], [2.5, 97.5])
                summary_rows.append({"model_name": m, "score": s, "n_seeds_valid": int(np.isfinite(pt).sum()),
                                     "auroc_mean": np.nanmean(pt), "auroc_sd_across_seeds": np.nanstd(pt, ddof=1),
                                     "boot_ci95_low": lo, "boot_ci95_high": hi,
                                     "boot_se": np.nanstd(seedmean_boot[s], ddof=1)})
            for a, b in DIFF_PAIRS:
                paired_diff = np.array(point[a], dtype=float) - np.array(point[b], dtype=float)
                diff_boot = seedmean_boot[a] - seedmean_boot[b]
                lo, hi = np.nanpercentile(diff_boot, [2.5, 97.5])
                diff_rows.append({"model_name": m, "comparison": f"{a} - {b}",
                                  "diff_mean": np.nanmean(paired_diff),
                                  "diff_sd_across_seeds": np.nanstd(paired_diff, ddof=1),
                                  "boot_ci95_low": lo, "boot_ci95_high": hi,
                                  "ci_excludes_zero": bool(lo > 0 or hi < 0)})
    return pd.DataFrame(seed_rows), pd.DataFrame(summary_rows), pd.DataFrame(diff_rows)


# ---- run everything ----------------------------------------------------------------------
def run_selective_analysis(store, seeds_by_model, out_xlsx=OUT_XLSX, verbose=True):
    rows = []
    for m, seeds in seeds_by_model.items():
        for seed in seeds:
            rows += selective_rows_for_seed(m, seed, store[(m, seed)])
    df_seed = pd.DataFrame(rows)
    df_fmt, df_num = summarize_selective(df_seed)
    df_auc_seed, df_auc_sum, df_auc_diff = auroc_analysis(store, seeds_by_model)

    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        df_seed.to_excel(writer, sheet_name="selective_per_seed", index=False)
        df_fmt.to_excel(writer, sheet_name="selective_mean_sd", index=False)
        df_num.to_excel(writer, sheet_name="selective_numeric", index=False)
        df_auc_seed.to_excel(writer, sheet_name="auroc_per_seed", index=False)
        df_auc_sum.to_excel(writer, sheet_name="auroc_ci_bootstrap", index=False)
        df_auc_diff.to_excel(writer, sheet_name="auroc_score_differences", index=False)

    if verbose:
        sel = df_fmt[(df_fmt.policy == "target_coverage") & (np.isclose(df_fmt.target, PRIMARY_COVERAGE))]
        print(f"\n===== SELECTIVE PREDICTION | target validation coverage = {PRIMARY_COVERAGE} "
              f"(mean ± SD, {df_fmt.n_seeds.max()} seeds) =====")
        print(sel[["model_name", "score", "coverage", "selective_acc", "errors_rejected",
                   "error_rejection_rate"]].to_string(index=False))
        print("\n===== ERROR-DETECTION AUROC (mean across seeds, 95% CI from test-image bootstrap) =====")
        t = df_auc_sum.copy()
        t["auroc_CI95"] = [f"{a:.4f} [{lo:.4f}, {hi:.4f}]"
                           for a, lo, hi in zip(t.auroc_mean, t.boot_ci95_low, t.boot_ci95_high)]
        print(t[["model_name", "score", "auroc_CI95"]].to_string(index=False))
        print(f"\nSaved: {out_xlsx}")
    return df_seed, df_fmt, df_auc_sum, df_auc_diff

### 3.3.2 Running the analysis

In [ ]:
extract_all_scores(MODEL_NAMES)                     # Stage 1 (heavy; finished seeds are skipped)
store, seeds_by_model = load_store(MODEL_NAMES)
df_sel_seed, df_sel_fmt, df_auc_sum, df_auc_diff = run_selective_analysis(store, seeds_by_model)   # Stages 2 + 3

## 3.4 Deterministic vs. standard dropout vs. MC Dropout

The 30 trained ResNet18 checkpoints are evaluated on the test set in three inference modes:

| Mode | Dropout at inference | Passes | Source of the numbers |
|---|---|---|---|
| `deterministic` | OFF (identity) | 1 | computed here |
| `dropout` | ON (standard dropout, one random mask) | 1 | computed here |
| `mc_dropout` | ON | T = 50, probabilities averaged | read from `results_30_seeds_resnet18.xlsx` (Section 3.1) |

BatchNorm layers are in evaluation mode in all three modes. The run seed is re-applied before each evaluation so the stochastic `dropout` mode is repeatable.

*Naming note:* the AURC of Section 3.1 (`test_aurc`) is stored as `test_aucr` in the output of this section; both denote the same quantity (area under the risk–coverage curve).

**Inputs:** `df_test.csv`, `results_30_seeds_resnet18.xlsx`, `saved_models/resnet18/resnet18_seed_*.pth`. **Output:** `three_mode_comparison_resnet18.xlsx` (30 seeds × 3 modes = 90 rows) and a printed mean ± SD per mode.

In [ ]:
# (key from compute_all_metrics, column in the 3-mode file, column in results_30_seeds_<model>.xlsx)
METRIC_COLUMNS = [
    ("acc", "test_accuracy", "test_acc"),
    ("roc_auc", "test_auc_roc", "test_roc_auc"),
    ("macro_f1", "test_f1_score", "test_macro_f1"),
    ("macro_precision", "test_precision", "test_macro_precision"),
    ("macro_recall", "test_recall", "test_macro_recall"),
    ("macro_specificity", "test_specificity", "test_macro_specificity"),
    ("ece", "test_ece", "test_ece"),
    ("mce", "test_mce", "test_mce"),
    ("brier", "test_brier", "test_brier"),
    ("nll", "test_nll", "test_nll"),
    ("aurc", "test_aucr", "test_aurc"),
]
THREE_MODE_XLSX = "three_mode_comparison_resnet18.xlsx"


@torch.no_grad()
def evaluate_two_modes(model, test_loader):
    """Evaluates the deterministic network (dropout OFF) and standard dropout (ON, one pass)."""
    results = {}
    dropout_layer = model.fc[0]                  # MCDropout layer in front of the classifier
    for mode, enabled in (("deterministic", False), ("dropout", True)):
        model.eval()                             # BatchNorm in eval mode in both cases
        dropout_layer.enabled = enabled
        all_probs, all_labels = [], []
        for images, labels in test_loader:
            logits = model(images.to(device))
            all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
            all_labels.append(labels.numpy())
        metrics = compute_all_metrics(np.vstack(all_probs), np.concatenate(all_labels))
        results[mode] = {three_col: metrics[key] for key, three_col, _ in METRIC_COLUMNS}
    dropout_layer.enabled = True                 # restore the default MC behaviour
    return results


def run_three_mode_comparison(output_xlsx=THREE_MODE_XLSX):
    config_files = glob.glob(f"results_30_seeds_{FOCUS_MODEL}.*")
    if not config_files:
        print(f"Configuration file 'results_30_seeds_{FOCUS_MODEL}' not found.")
        return None
    print(f"Reading configuration from: {config_files[0]}")
    df_config = pd.read_excel(config_files[0])
    df_config.columns = df_config.columns.str.strip().str.lower()

    model_files = sorted(glob.glob(f"{SAVED_MODELS_DIR}/{FOCUS_MODEL}/{FOCUS_MODEL}_seed_*.pth"))
    if not model_files:
        print(f"No {FOCUS_MODEL} checkpoints found in {SAVED_MODELS_DIR}/{FOCUS_MODEL}/")
        return None
    print(f"Found {len(model_files)} {FOCUS_MODEL} checkpoints.")
    print("Evaluation: deterministic + standard dropout (MC Dropout values come from the configuration file)\n")

    all_rows = []
    for model_path in model_files:
        seed = int(os.path.basename(model_path).split("_seed_")[1].split(".")[0])
        match = df_config[df_config["seed"] == seed]
        if match.empty:
            print(f"Warning: seed {seed} not found in the configuration file. Skipped.")
            continue
        cfg = match.iloc[0]
        dropout_rate = float(cfg.get("dropout_rate", 0.5))
        batch_size = int(cfg.get("batch_size", 16))
        train_acc = cfg.get("train_acc", cfg.get("train_accuracy", np.nan))
        print(f"Seed {seed:4d} | dropout: {dropout_rate:.3f} | train acc: {train_acc:.4f}")

        set_seed(seed)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)
        model = get_model(FOCUS_MODEL, dropout_rate=dropout_rate, pretrained=False).to(device)
        model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
        eval_results = evaluate_two_modes(model, test_loader)

        common = {"model_name": FOCUS_MODEL, "seed": seed, "dropout_rate": dropout_rate,
                  "batch_size": batch_size, "train_accuracy": train_acc}
        for mode in ("deterministic", "dropout"):
            all_rows.append({**common, "mode": mode, **eval_results[mode]})
        # MC Dropout (T = 50): already computed in Section 3.1
        all_rows.append({**common, "mode": "mc_dropout",
                         **{three_col: cfg.get(cfg_col, np.nan) for _, three_col, cfg_col in METRIC_COLUMNS}})

    df_final = pd.DataFrame(all_rows)
    df_final.to_excel(output_xlsx, index=False)
    print(f"\nDone. Detailed results saved to: {output_xlsx}")

    print("\n" + "=" * 75)
    print(" MEAN ± SD PER MODE (SAMPLE SD ACROSS SEEDS)")
    print("=" * 75)
    metrics_display = ["train_accuracy"] + [three_col for _, three_col, _ in METRIC_COLUMNS]
    for mode in ("deterministic", "dropout", "mc_dropout"):
        print(f"\n[{mode.upper().replace('_', ' ')}]")
        mode_df = df_final[df_final["mode"] == mode]
        for metric in metrics_display:
            clean_name = metric.replace("test_", "").replace("_", " ").title()
            print(f"   {clean_name:<18}: {mode_df[metric].mean():.4f} ± {mode_df[metric].std():.4f}")
    return df_final

In [ ]:
df_three_modes = run_three_mode_comparison()

## 3.5 Deterministic vs. MC Dropout — paired test

For ECE, NLL and Brier score, the 30 paired runs (same seed) of the deterministic and the MC Dropout mode are compared with a **two-sided Wilcoxon signed-rank test**. Printed for each metric: the mean ± SD of both modes, the Wilcoxon statistic and the p-value.

*Note:* the SD shown here is the **population SD** (`ddof = 0`), whereas Section 3.4 prints the sample SD (`ddof = 1`).

**Input:** the dataframe of Section 3.4 (or `three_mode_comparison_resnet18.xlsx`). **Output:** printed table only.

In [ ]:
def wilcoxon_deterministic_vs_mc(df_modes, metrics=("test_ece", "test_nll", "test_brier")):
    print("=" * 75)
    print(" PAIRED WILCOXON TEST: DETERMINISTIC vs MC DROPOUT")
    print(f"     Metrics: {', '.join(m.replace('test_', '').upper() for m in metrics)}")
    print("=" * 75)

    df_det = df_modes[df_modes["mode"] == "deterministic"].set_index("seed")
    df_mc = df_modes[df_modes["mode"] == "mc_dropout"].set_index("seed")
    common_seeds = df_det.index.intersection(df_mc.index)        # only seeds that have both modes
    df_det = df_det.loc[common_seeds].sort_index()
    df_mc = df_mc.loc[common_seeds].sort_index()

    rows = []
    for metric in metrics:
        x, y = df_det[metric].values, df_mc[metric].values
        mask = ~np.isnan(x) & ~np.isnan(y)                       # drop incomplete pairs
        x_clean, y_clean = x[mask], y[mask]
        if len(x_clean) < 3:
            print(f"\n   Not enough data for {metric} (only {len(x_clean)} valid seeds).")
            continue

        stat, p_value = stats.wilcoxon(x_clean, y_clean, zero_method="wilcox", alternative="two-sided")
        verdict = "Significant (p < 0.05)" if p_value < 0.05 else "Not significant (p >= 0.05)"
        print(f"\n[{metric.replace('test_', '').upper()}]")
        print(f"   Deterministic mean : {x_clean.mean():.4f} ± {x_clean.std():.4f}")
        print(f"   MC Dropout mean    : {y_clean.mean():.4f} ± {y_clean.std():.4f}")
        print(f"   Wilcoxon statistic : {stat:.4f}")
        print(f"   p-value            : {p_value:.4e}  {verdict}")
        rows.append({"metric": metric, "n_pairs": len(x_clean), "wilcoxon_stat": stat, "p_value": p_value})
    return pd.DataFrame(rows)

In [ ]:
if df_three_modes is None:
    df_three_modes = pd.read_excel(THREE_MODE_XLSX)
df_wilcoxon = wilcoxon_deterministic_vs_mc(df_three_modes)

## 3.6 MC Dropout convergence

How many stochastic passes are needed? The 30 ResNet18 checkpoints are run with T = 50 passes on the test set, and accuracy, mean predictive entropy, mean predictive variance and ECE are recomputed using only the first $T$ passes for $T \in \{1, 2, 3, 5, 10, 15, 20, 30, 40, 50\}$.

* *Mean predictive entropy* – average over test images of $H[\bar p]$.
* *Mean predictive variance* – variance of the class probabilities across the $T$ passes, averaged over classes and images.
* Mean ± SD across the 30 seeds (population SD, `ddof = 0`).

**Inputs:** `df_test.csv`, `results_30_seeds_resnet18.xlsx`, ResNet18 checkpoints. **Outputs:** `mc_dropout_convergence_summary.csv`, `mc_dropout_convergence.png` (4 panels) and a printed summary table.

In [ ]:
MC_PASSES_LIST = [1, 2, 3, 5, 10, 15, 20, 30, 40, 50]


def mean_predictive_entropy(probs, eps=1e-12):
    """Mean over samples of the predictive entropy of the averaged probabilities."""
    return np.mean(-np.sum(probs * np.log(probs + eps), axis=1))


def mean_predictive_variance(mc_probs):
    """Variance across MC passes, averaged over classes and samples; mc_probs: (N, T, C)."""
    return np.mean(np.var(mc_probs, axis=1))


def evaluate_model_all_T(model, dataloader, T_list):
    """One model, all T values: run the largest T once and use the first T passes for each smaller T."""
    mc_probs_all, labels_all = mc_dropout_predict(model, dataloader, mc_passes=max(T_list))   # (N, Tmax, C)
    results = {}
    for T in T_list:
        mc_probs_T = mc_probs_all[:, :T, :]
        mean_probs = mc_probs_T.mean(axis=1)
        results[T] = {
            "accuracy": np.mean(np.argmax(mean_probs, axis=1) == labels_all) * 100.0,
            "entropy": mean_predictive_entropy(mean_probs),
            "variance": mean_predictive_variance(mc_probs_T),
            "ece": compute_calibration_metrics(mean_probs, labels_all)["ece"],
        }
    return results


def run_convergence_analysis(model_name=FOCUS_MODEL, T_list=MC_PASSES_LIST):
    df_config = pd.read_excel(f"results_30_seeds_{model_name}.xlsx")
    config = df_config.iloc[0]
    dropout_rate = float(config["dropout_rate"])
    batch_size = int(config["batch_size"])
    print(f"Configuration: dropout_rate={dropout_rate}, lr={config['lr']}, "
          f"batch_size={batch_size}, optimizer={config['optimizer']}")

    model_files = sorted(glob.glob(os.path.join(SAVED_MODELS_DIR, model_name, f"{model_name}_seed_*.pth")))
    print(f"Found {len(model_files)} models in {SAVED_MODELS_DIR}/{model_name}")
    if not model_files:
        raise FileNotFoundError(f"No models found in {SAVED_MODELS_DIR}/{model_name}")

    all_results = {T: {"accuracy": [], "entropy": [], "variance": [], "ece": []} for T in T_list}
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

    for idx, model_path in enumerate(model_files, 1):
        print(f"[{idx:2d}/{len(model_files)}] Loading {os.path.basename(model_path)} ...")
        seed = int(os.path.basename(model_path).split("_seed_")[1].split(".")[0])
        set_seed(seed)    # fixes the MC dropout masks of this run

        model = get_model(model_name, dropout_rate=dropout_rate, pretrained=False).to(device)
        model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
        results = evaluate_model_all_T(model, test_loader, T_list)
        for T in T_list:
            for key in all_results[T]:
                all_results[T][key].append(results[T][key])
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

    summary = {}
    for T in T_list:
        summary[T] = {}
        for key in ("accuracy", "entropy", "variance", "ece"):
            summary[T][f"{key}_mean"] = np.mean(all_results[T][key])
            summary[T][f"{key}_std"] = np.std(all_results[T][key])       # population SD (ddof = 0)

    df_summary = pd.DataFrame(summary).T
    df_summary.index.name = "T"
    print("\n=== Summary (mean ± SD across seeds) ===")
    print(df_summary.to_string())
    df_summary.to_csv("mc_dropout_convergence_summary.csv")
    print("\nSaved to mc_dropout_convergence_summary.csv")
    return all_results, summary


def plot_convergence(summary, T_list=MC_PASSES_LIST, title="MC Dropout Convergence Analysis - ResNet18 (30 Seeds)",
                     output_path="mc_dropout_convergence.png"):
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
    panels = [
        ("accuracy", "Accuracy (%)", "Accuracy Convergence", colors[0]),
        ("entropy", "Mean Predictive Entropy", "Entropy Convergence", colors[1]),
        ("variance", "Mean Predictive Variance", "Variance Convergence", colors[2]),
        ("ece", "Expected Calibration Error", "Calibration Convergence", colors[3]),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(title, fontsize=16, fontweight="bold", y=1.01)

    for ax, (key, ylabel, panel_title, color) in zip(axes.flatten(), panels):
        means = np.array([summary[T][f"{key}_mean"] for T in T_list])
        stds = np.array([summary[T][f"{key}_std"] for T in T_list])
        ax.plot(T_list, means, "o-", color=color, linewidth=2, markersize=6, label="Mean")
        ax.fill_between(T_list, means - stds, means + stds, alpha=0.2, color=color, label="±1 Std")
        ax.set_xlabel("Number of MC Passes (T)", fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(panel_title, fontsize=13, fontweight="bold")
        ax.set_xticks(T_list)
        ax.grid(True, alpha=0.3, linestyle="--")
        ax.legend(loc="best", fontsize=10)
        for T, m in zip(T_list, means):                      # annotate the mean of every point
            ax.annotate(f"{m:.4f}", xy=(T, m), xytext=(0, 8), textcoords="offset points",
                        ha="center", fontsize=7, color=color)

    plt.tight_layout()
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    print(f"\nFigure saved to: {output_path}")
    plt.show()

In [ ]:
print("=" * 70)
print("MC DROPOUT CONVERGENCE ANALYSIS")
print(f"Model: {FOCUS_MODEL} | Seeds: {N_SEEDS} | T: {MC_PASSES_LIST}")
print("=" * 70)
convergence_results, convergence_summary = run_convergence_analysis()
plot_convergence(convergence_summary)

## 3.7 Per-class metrics, confusion matrix and reliability diagram

Detailed evaluation of one run (**ResNet18, seed 360**) with MC Dropout (T = 50) on the test set. The seed is one of the 30 seeds of Section 3.1; the criterion used to pick it is not encoded in this notebook and should be stated in the manuscript.

Three figures are produced (saved at 600 dpi in `plots/` and also shown inline):

1. **Per-class metrics** – precision, recall, F1, accuracy, specificity and AUC-ROC of every class (one-vs-rest; the per-class accuracy is $(TP+TN)/N$).
2. **Confusion matrix** – raw counts (rows = true class, columns = predicted class).
3. **Reliability diagram** – accuracy as a function of confidence in 15 equal-width bins, with the diagonal as perfect calibration, the bin counts as bars, and the ECE in the title. Empty bins are not drawn on the curve.

**Inputs:** `df_test.csv`, `results_30_seeds_resnet18.xlsx`, `saved_models/resnet18/resnet18_seed_360.pth`. **Outputs:** `plots/resnet18_seed360_per_class_metrics.png`, `plots/resnet18_seed360_confusion_matrix.png`, `plots/resnet18_seed360_reliability_diagram.png`.

In [ ]:
def plot_per_class_metrics(y_true, y_pred, y_probs, class_names, save_path=None):
    n_classes = len(class_names)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, labels=range(n_classes), zero_division=0)

    cm = confusion_matrix(y_true, y_pred, labels=range(n_classes))
    specificity = specificity_per_class(cm)
    accuracy = np.array([(cm[i, i] + (cm.sum() - cm[i, :].sum() - cm[:, i].sum() + cm[i, i])) / cm.sum()
                         for i in range(n_classes)])                     # (TP + TN) / N
    auc_roc = []
    for i in range(n_classes):
        try:
            auc_roc.append(roc_auc_score((y_true == i).astype(int), y_probs[:, i]))
        except ValueError:
            auc_roc.append(0.0)

    series = [("Precision", precision, "#4C72B0"), ("Recall", recall, "#55A868"),
              ("F1-Score", f1, "#C44E52"), ("Accuracy", accuracy, "#8172B2"),
              ("Specificity", specificity, "#D4A017"), ("AUC ROC", np.array(auc_roc), "#2E86C1")]
    offsets = [-0.30, -0.18, -0.06, 0.06, 0.18, 0.30]
    width = 0.12
    x = np.arange(n_classes)

    fig, ax = plt.subplots(figsize=(14, 7))
    for (label, values, color), offset in zip(series, offsets):
        bars = ax.bar(x + offset, values, width, label=label, color=color, edgecolor="white", linewidth=0.5)
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f"{bar.get_height():.4f}",
                    ha="center", va="bottom", fontsize=8, fontweight="bold", color="black", rotation=45)

    ax.set_ylabel("Score", fontsize=14)
    ax.set_title("Per-Class Precision, Recall, F1-Score, Accuracy, Specificity, and AUC ROC",
                 fontsize=15, fontweight="bold", pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=0, ha="center", fontsize=13, fontweight="bold")
    ax.set_ylim(0, 1.20)
    ax.legend(fontsize=11, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=6, frameon=True)
    ax.grid(axis="y", linestyle="--", alpha=0.5, zorder=0)
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=600, bbox_inches="tight")
        print(f"Per-class metrics plot saved: {save_path}")
    plt.show()


def plot_confusion_matrix(y_true, y_pred, class_names, save_path=None, normalize=False):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    if normalize:
        cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
        fmt, title = ".4f", "Normalized Confusion Matrix"
    else:
        fmt, title = "d", "Confusion Matrix"

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues, aspect="auto")
    ax.set_title(title, fontsize=16, fontweight="bold", pad=20)
    ax.set_xlabel("Predicted Label", fontsize=14, labelpad=10)
    ax.set_ylabel("True Label", fontsize=14, labelpad=10)
    ticks = np.arange(len(class_names))
    ax.set_xticks(ticks)
    ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=12, fontweight="bold")
    ax.set_yticks(ticks)
    ax.set_yticklabels(class_names, fontsize=12, fontweight="bold")

    threshold = cm.max() / 2.0                      # switch the text colour on dark cells
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt), ha="center", va="center",
                    color="white" if cm[i, j] > threshold else "black", fontsize=14, fontweight="bold")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=600, bbox_inches="tight")
        print(f"Confusion matrix saved: {save_path}")
    plt.show()


def plot_reliability_diagram(y_true, y_probs, n_bins=10, save_path=None):
    preds = np.argmax(y_probs, axis=1)
    confidences = np.max(y_probs, axis=1)
    accuracies = (preds == y_true).astype(float)

    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_accs, bin_confs, bin_counts = [], [], []
    for i in range(n_bins):
        in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
        prop_in_bin = np.mean(in_bin)
        if prop_in_bin > 0:
            acc_in_bin = np.mean(accuracies[in_bin])
            conf_in_bin = np.mean(confidences[in_bin])
            ece += np.abs(acc_in_bin - conf_in_bin) * prop_in_bin
            bin_accs.append(acc_in_bin)
            bin_confs.append(conf_in_bin)
            bin_counts.append(np.sum(in_bin))
        else:                                        # empty bin: placed at the bin centre, not drawn on the curve
            bin_accs.append(np.nan)
            bin_confs.append((bin_boundaries[i] + bin_boundaries[i + 1]) / 2)
            bin_counts.append(0)
    bin_accs, bin_confs, bin_counts = np.array(bin_accs), np.array(bin_confs), np.array(bin_counts)
    non_empty = bin_counts > 0

    fig, ax1 = plt.subplots(figsize=(8, 6))
    ax1.plot(bin_confs[non_empty], bin_accs[non_empty], marker="o", linestyle="-", color="#D65F5F",
             linewidth=2, markersize=8, label="Model Calibration")
    ax1.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=2, label="Perfectly Calibrated")
    ax1.set_xlabel("Mean Predicted Probability (Confidence)", fontsize=12)
    ax1.set_ylabel("Fraction of Positives (Accuracy)", fontsize=12)
    ax1.set_title(f"Reliability Diagram (ECE = {ece:.4f})", fontsize=14, fontweight="bold")
    ax1.set_xlim(0, 1.05)
    ax1.set_ylim(0, 1.05)
    ax1.grid(True, linestyle="--", alpha=0.6)

    ax2 = ax1.twinx()
    ax2.bar(bin_confs, bin_counts, width=0.05, alpha=0.4, color="#4C72B0", label="Confidence Distribution")
    ax2.set_ylabel("Frequency (Count)", fontsize=12)
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(handles2 + handles1, labels2 + labels1, loc="upper left", fontsize=11, framealpha=0.9)

    max_count = max(bin_counts) if len(bin_counts) > 0 else 1
    for conf, count in zip(bin_confs, bin_counts):
        if count > 0:
            ax2.text(conf, count + max_count * 0.02, f"{int(count)}", ha="center", va="bottom",
                     fontsize=7, fontweight="bold", color="#2c3e50")
    ax2.set_ylim(0, max_count * 1.15)
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=600, bbox_inches="tight")
        print(f"Reliability diagram saved: {save_path}")
    plt.show()


def plot_specific_saved_model(model_name=FOCUS_MODEL, seed=FOCUS_SEED, class_names=CLASS_NAMES,
                              mc_passes=MC_PASSES):
    """Loads one checkpoint, runs MC Dropout on the test set and draws the three figures."""
    excel_path = f"results_30_seeds_{model_name}.xlsx"
    model_path = f"{SAVED_MODELS_DIR}/{model_name}/{model_name}_seed_{seed}.pth"
    if not os.path.exists(excel_path):
        print(f"File {excel_path} not found. Make sure the training of Section 3.1 has finished.")
        return
    if not os.path.exists(model_path):
        print(f"Model file {model_path} not found.")
        return

    df = pd.read_excel(excel_path)
    row = df[df["seed"] == seed]
    if row.empty:
        print(f"Seed {seed} not found in {excel_path}.")
        return
    dropout_rate = float(row["dropout_rate"].values[0])
    print(f"\n================ LOADING MODEL: {model_name.upper()} (seed: {seed}, dropout: {dropout_rate}) ================")

    set_seed(seed)
    model = get_model(model_name, dropout_rate=dropout_rate, pretrained=False).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, pin_memory=True)

    print("Running MC Dropout evaluation on the test set ...")
    test_mc_probs, test_labels = mc_dropout_predict(model, test_loader, mc_passes=mc_passes)
    mean_probs = test_mc_probs.mean(axis=1)
    preds = np.argmax(mean_probs, axis=1)

    os.makedirs(PLOTS_DIR, exist_ok=True)
    prefix = f"{PLOTS_DIR}/{model_name}_seed{seed}"
    plot_per_class_metrics(test_labels, preds, mean_probs, class_names, save_path=f"{prefix}_per_class_metrics.png")
    plot_confusion_matrix(test_labels, preds, class_names, save_path=f"{prefix}_confusion_matrix.png")
    plot_reliability_diagram(test_labels, mean_probs, n_bins=15, save_path=f"{prefix}_reliability_diagram.png")
    print(f"\nDone. 3 figures for {model_name} seed {seed} saved in '{PLOTS_DIR}/'.")

In [ ]:
plot_specific_saved_model(model_name=FOCUS_MODEL, seed=FOCUS_SEED)

## 3.8 Uncertainty-based error detection

*Does the model's uncertainty reveal where it is wrong?* For **ResNet18, seed 360** (MC Dropout, T = 50, test set) the predictive uncertainty is decomposed into total, aleatoric and epistemic parts (Section 0.7) and visualised in two figures:

1. **Uncertainty distribution per tumor class** – box plots of total entropy, aleatoric and epistemic uncertainty for each true class.
2. **Correct vs. wrong predictions** – (a) scatter of total entropy against confidence, (b) histogram of aleatoric uncertainty, (c) histogram of epistemic uncertainty (mutual information), each split by prediction correctness.

A printed table gives, per class, the number of correct / wrong predictions and the mean ± SD of the three uncertainty measures.

**Inputs:** `df_test.csv`, `results_30_seeds_resnet18.xlsx` (dropout rate), the seed-360 checkpoint. **Outputs:** `resnet18_seed360_uncertainty_distribution_per_class.png`, `resnet18_seed360_uncertainty_correct_vs_wrong_analysis.png`, printed summary.

In [ ]:
DISPLAY_CLASS_NAMES = ["Glioma", "Meningioma", "No tumor", "Pituitary"]


def run_uncertainty_visualisation(model_name=FOCUS_MODEL, seed=FOCUS_SEED, batch_size=32):
    model_path = f"{SAVED_MODELS_DIR}/{model_name}/{model_name}_seed_{seed}.pth"
    dropout_rate = BEST_CONFIGS[model_name]["dropout_rate"]     # must equal the rate used in training

    set_seed(seed)
    model = get_model(model_name, dropout_rate=dropout_rate, pretrained=False).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    print(f"[OK] Model loaded: {model_path} (dropout = {dropout_rate})")

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)
    print(f"[INFO] Running MC Dropout inference (T={MC_PASSES}) ...")
    mc_probs, labels = mc_dropout_predict(model, test_loader, mc_passes=MC_PASSES)
    mean_probs = mc_probs.mean(axis=1)
    preds = np.argmax(mean_probs, axis=1)
    unc = decompose_uncertainty(mc_probs)
    print(f"[OK] Total samples: {len(labels)}")

    df_unc = pd.DataFrame({
        "class": [DISPLAY_CLASS_NAMES[y] for y in labels],
        "total_entropy": unc["total"],
        "aleatoric": unc["aleatoric"],
        "epistemic": unc["epistemic"],
        "confidence": np.max(mean_probs, axis=1),
        "correct": (preds == labels).astype(int),
    })
    prefix = f"{model_name}_seed{seed}"

    # ---- Figure 1: uncertainty distribution per class ----
    sns.set_style("whitegrid")
    fig1, axes1 = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (col, title) in zip(axes1, [("total_entropy", "Total Predictive Entropy"),
                                        ("aleatoric", "Aleatoric Uncertainty"),
                                        ("epistemic", "Epistemic Uncertainty (MI)")]):
        sns.boxplot(data=df_unc, x="class", y=col, ax=ax, color="lightgray", width=0.5,
                    flierprops=dict(marker="o", markersize=4, markerfacecolor="white",
                                    markeredgecolor="black", markeredgewidth=0.8))
        ax.set_title(title, fontsize=13, fontweight="bold")
        ax.set_xlabel("Class", fontsize=11)
        ax.set_ylabel("Entropy (nats)", fontsize=11)
        ax.tick_params(axis="x", rotation=30)
    fig1.suptitle("Uncertainty Distribution per Brain Tumor Class", fontsize=15, fontweight="bold", y=1.02)
    fig1.tight_layout()
    path1 = f"{prefix}_uncertainty_distribution_per_class.png"
    fig1.savefig(path1, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"[OK] Figure 1 saved as: {path1}")

    # ---- Figure 2: correct vs. wrong predictions ----
    correct_mask = df_unc["correct"] == 1
    wrong_mask = ~correct_mask
    fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))

    ax = axes2[0]
    ax.scatter(df_unc.loc[correct_mask, "total_entropy"], df_unc.loc[correct_mask, "confidence"],
               c="green", alpha=0.5, s=30, label="Correct", edgecolors="none")
    ax.scatter(df_unc.loc[wrong_mask, "total_entropy"], df_unc.loc[wrong_mask, "confidence"],
               c="red", alpha=0.7, s=40, label="Wrong", edgecolors="none")
    ax.set_xlabel("Total Predictive Entropy", fontsize=11)
    ax.set_ylabel("Confidence", fontsize=11)
    ax.set_title("Total Entropy vs Confidence", fontsize=13, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # Histograms share bin edges for both groups; the upper limit keeps the original range
    # (0.7 / 0.175) unless the data exceed it, so that no sample is cut off.
    for ax, col, label, base_limit in [(axes2[1], "aleatoric", "Aleatoric Uncertainty", 0.7),
                                       (axes2[2], "epistemic", "Epistemic Uncertainty (MI)", 0.175)]:
        bins = np.linspace(0, max(base_limit, df_unc[col].max()), 30)
        ax.hist(df_unc.loc[correct_mask, col], bins=bins, color="green", alpha=0.6, label="Correct", edgecolor="white")
        ax.hist(df_unc.loc[wrong_mask, col], bins=bins, color="red", alpha=0.6, label="Wrong", edgecolor="white")
        ax.set_xlabel(label, fontsize=11)
        ax.set_ylabel("Frequency", fontsize=11)
        ax.set_title(f"{col.capitalize()} Distribution", fontsize=13, fontweight="bold")
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)

    fig2.suptitle("Uncertainty Analysis: Correct vs. Wrong Predictions", fontsize=15, fontweight="bold", y=1.02)
    fig2.tight_layout()
    path2 = f"{prefix}_uncertainty_correct_vs_wrong_analysis.png"
    fig2.savefig(path2, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"[OK] Figure 2 saved as: {path2}")

    # ---- Summary table ----
    print("\n" + "=" * 70)
    print(f"UNCERTAINTY SUMMARY - {model_name} seed {seed}")
    print("=" * 70)
    for cls in DISPLAY_CLASS_NAMES:
        sub = df_unc[df_unc["class"] == cls]
        n_correct = int(sub["correct"].sum())
        print(f"\n[{cls}]  N={len(sub)}  (Correct={n_correct}, Wrong={len(sub) - n_correct})")
        for col, label in [("total_entropy", "Total Entropy "), ("aleatoric", "Aleatoric     "),
                           ("epistemic", "Epistemic (MI)")]:
            print(f"  {label} -> mean={sub[col].mean():.4f}  std={sub[col].std():.4f}")
    print("\n[DONE] Both figures created.")
    return df_unc

In [ ]:
df_uncertainty = run_uncertainty_visualisation()

## 3.9 Uncertainty threshold and Grad-CAM

Selective prediction needs a rejection threshold. Here the threshold is derived on the **validation set** and then applied **unchanged** to the test set (ResNet18, seed 360, MC Dropout T = 50, epistemic uncertainty = mutual information).

| Step | Description | Data |
|---|---|---|
| 3.9.1 | Threshold search: candidate thresholds are the 1st … 99th percentiles of the validation uncertainty; among candidates with coverage ≥ 90 % the one with the highest selective accuracy is chosen | validation |
| 3.9.2 | MC Grad-CAM for every test image (50 stochastic Grad-CAM maps → mean map and SD map) together with the uncertainty of every image | test |
| 3.9.3 | The fixed threshold of 3.9.1 is applied to the test results: coverage, selective accuracy, accuracy of the rejected samples, and three distribution plots | test |

### 3.9.1 Threshold search on the validation set

**Inputs:** `df_validation.csv`, seed-360 checkpoint. **Outputs:** `epistemic_threshold_analysis.png` (uncertainty histogram with the chosen threshold; coverage-vs-selective-accuracy curve), `threshold_search_log_resnet18_360.csv` (one row per candidate threshold) and a printed result block.

In [ ]:
def find_optimal_threshold(epistemic_unc, preds, labels, min_coverage=0.90):
    """Threshold on epistemic uncertainty.

    Candidates are the 1st ... 99th percentiles of the validation uncertainty. A prediction is
    accepted if uncertainty <= threshold. Among the candidates with coverage >= `min_coverage`
    the one with the highest selective accuracy is selected.
    """
    is_correct = (preds == labels).astype(int)
    thresholds = np.percentile(epistemic_unc, np.arange(1, 100))

    log = []
    for thresh in thresholds:
        accepted = epistemic_unc <= thresh
        coverage = np.mean(accepted)
        if coverage == 0:
            continue
        log.append({
            "threshold": thresh,
            "coverage": coverage,
            "selective_acc": np.mean(is_correct[accepted]),
            # F1 of "accepted" as a predictor of "correct"
            "f1_correct": f1_score(is_correct, accepted.astype(int), zero_division=0),
        })
    df_search_log = pd.DataFrame(log)

    valid = df_search_log[df_search_log["coverage"] >= min_coverage]
    if not valid.empty:
        best_row = valid.loc[valid["selective_acc"].idxmax()]
    else:                                   # fallback: no candidate reaches the required coverage
        best_row = df_search_log.iloc[-1]

    best_metrics = {k: best_row[k] for k in ("threshold", "coverage", "selective_acc", "f1_correct")}
    return best_row["threshold"], best_metrics, df_search_log


def run_epistemic_threshold_search(model_name=FOCUS_MODEL, seed=FOCUS_SEED, mc_passes=MC_PASSES):
    print("\n" + "=" * 70)
    print("FINDING THE BEST EPISTEMIC UNCERTAINTY THRESHOLD")
    print(f"Model: {model_name} | Seed: {seed} | Data: Validation")
    print("=" * 70)

    set_seed(seed)
    dropout_rate = BEST_CONFIGS[model_name]["dropout_rate"]
    model_path = f"{SAVED_MODELS_DIR}/{model_name}/{model_name}_seed_{seed}.pth"
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found at {model_path}. Please ensure training is complete.")
    model = get_model(model_name, dropout_rate=dropout_rate, pretrained=False).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    print(f"[INFO] Loaded model from: {model_path}")

    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, pin_memory=True)
    print(f"[INFO] Running MC Dropout (T={mc_passes}) on validation data ...")
    mc_probs, labels = mc_dropout_predict(model, val_loader, mc_passes=mc_passes)
    preds = np.argmax(mc_probs.mean(axis=1), axis=1)

    print("[INFO] Calculating epistemic uncertainty (mutual information) ...")
    epistemic_unc = decompose_uncertainty(mc_probs)["epistemic"]

    print("[INFO] Searching for the optimal threshold ...")
    best_thresh, best_metrics, df_search_log = find_optimal_threshold(epistemic_unc, preds, labels)

    print("\n" + "-" * 50)
    print("RESULTS OF THE BEST THRESHOLD SEARCH:")
    print("-" * 50)
    print(f"Epistemic uncertainty threshold : {best_metrics['threshold']:.5f}")
    print(f"Coverage (accepted data)        : {best_metrics['coverage'] * 100:.2f}%")
    print(f"Selective accuracy (accepted)   : {best_metrics['selective_acc'] * 100:.2f}%")
    print(f"F1-score (correct pred. class)  : {best_metrics['f1_correct']:.4f}")
    print(f"Global accuracy (no rejection)  : {np.mean(preds == labels) * 100:.2f}%")
    print("-" * 50)

    # ---- plots (both histograms share the same bin edges) ----
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    bins = np.linspace(epistemic_unc.min(), epistemic_unc.max(), 31)
    axes[0].hist(epistemic_unc[preds == labels], bins=bins, alpha=0.6, label="Correct Predictions", color="green")
    axes[0].hist(epistemic_unc[preds != labels], bins=bins, alpha=0.6, label="Incorrect Predictions", color="red")
    axes[0].axvline(best_thresh, color="black", linestyle="--", linewidth=2, label=f"Best Threshold ({best_thresh:.4f})")
    axes[0].set_xlabel("Epistemic Uncertainty (Mutual Information)")
    axes[0].set_ylabel("Frequency")
    axes[0].set_title("Uncertainty Distribution: Correct vs Incorrect")
    axes[0].legend()

    axes[1].plot(df_search_log["coverage"] * 100, df_search_log["selective_acc"] * 100, marker=".", linestyle="-")
    axes[1].scatter(best_metrics["coverage"] * 100, best_metrics["selective_acc"] * 100, color="red", s=100,
                    zorder=5, label=f"Best Threshold\nAcc: {best_metrics['selective_acc'] * 100:.1f}%")
    axes[1].set_xlabel("Coverage (%)")
    axes[1].set_ylabel("Selective Accuracy (%)")
    axes[1].set_title("Trade-off: Coverage vs Selective Accuracy")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.savefig("epistemic_threshold_analysis.png", dpi=300)
    print("[INFO] Plot saved as 'epistemic_threshold_analysis.png'")
    plt.show()

    log_csv = f"threshold_search_log_{model_name}_{seed}.csv"
    df_search_log.to_csv(log_csv, index=False)
    print(f"[INFO] Threshold search log saved to '{log_csv}'")
    return best_thresh, best_metrics, df_search_log

In [ ]:
validation_threshold, validation_threshold_metrics, df_threshold_log = run_epistemic_threshold_search()

### 3.9.2 MC Grad-CAM and per-image uncertainty on the test set

For every test image, 50 Grad-CAM maps (target layer `layer4`, target class = MC-averaged prediction) are generated with MC Dropout active; the **mean** map shows where the network typically looks, the **SD** map shows where that attention is unstable. The uncertainty (total / aleatoric / epistemic), confidence and correctness of every image are stored in one table.

**Inputs:** `df_test.csv`, seed-360 checkpoint. **Outputs:**

* `gradcam_outputs/resnet18_360_test/<image_id>/` with `original.png`, `mean_gradcam.png`, `std_gradcam.png` and the 50 single-pass maps `00.png … 49.png` (heavy: 52 files per test image),
* `test_analysis_resnet18_360_mc_gradcam.csv` (one row per test image: labels, correctness, confidence, uncertainties and file paths),
* a printed summary (accuracy, mean confidence / uncertainties, epistemic uncertainty of correct vs. incorrect predictions).

In [ ]:
class GradCAM:
    """Standard Grad-CAM for CNNs; works with MC Dropout because the model stays stochastic."""

    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        self._handles = [
            target_layer.register_forward_hook(self._save_activation),
            target_layer.register_full_backward_hook(self._save_gradient),
        ]

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def remove_hooks(self):
        for handle in self._handles:
            handle.remove()

    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        output = self.model(input_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1).item()

        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, target_class] = 1
        output.backward(gradient=one_hot)

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)          # global-average-pooled gradients
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)        # normalise to [0, 1]


def generate_mc_gradcams(gradcam, input_tensor, target_class, mc_passes=MC_PASSES):
    """`mc_passes` Grad-CAM maps with MC Dropout active; returns an array (mc_passes, H, W)."""
    return np.stack([gradcam.generate(input_tensor, target_class) for _ in range(mc_passes)], axis=0)


def save_gradcam_image(img_tensor, cam, save_path, alpha=0.5, colormap=cv2.COLORMAP_TURBO):
    """Overlays a heat map (TURBO: dark blue -> green -> yellow -> red) on the original image and saves it."""
    mean, std = np.array(IMAGENET_MEAN), np.array(IMAGENET_STD)
    img = img_tensor.squeeze().permute(1, 2, 0).cpu().numpy()
    img = (np.clip(std * img + mean, 0, 1) * 255).astype(np.uint8)          # de-normalise

    cam_min, cam_max = cam.min(), cam.max()                                   # use the full colour range
    cam = (cam - cam_min) / (cam_max - cam_min) if cam_max - cam_min > 1e-8 else np.zeros_like(cam)

    cam_resized = cv2.resize(cam, (img.shape[1], img.shape[0]))
    heatmap = cv2.cvtColor(cv2.applyColorMap(np.uint8(255 * cam_resized), colormap), cv2.COLOR_BGR2RGB)
    plt.imsave(save_path, cv2.addWeighted(img, 1 - alpha, heatmap, alpha, 0))


def run_test_analysis_and_mc_gradcam(model_name=FOCUS_MODEL, seed=FOCUS_SEED, mc_passes=MC_PASSES):
    dropout_rate = BEST_CONFIGS[model_name]["dropout_rate"]
    print("\n" + "=" * 70)
    print("TEST ANALYSIS & MC GRAD-CAM GENERATION")
    print(f"Model: {model_name} | Seed: {seed} | Dropout: {dropout_rate} | MC passes: {mc_passes}")
    print("=" * 70)

    set_seed(seed)
    model_path = f"{SAVED_MODELS_DIR}/{model_name}/{model_name}_seed_{seed}.pth"
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found at {model_path}.")
    model = get_model(model_name, dropout_rate=dropout_rate, pretrained=False).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    print(f"[INFO] Loaded model from: {model_path}")

    base_dir = f"gradcam_outputs/{model_name}_{seed}_test"
    os.makedirs(base_dir, exist_ok=True)

    # fast batched MC Dropout for predictions and uncertainties; batch size 1 for Grad-CAM
    test_loader_mc = DataLoader(test_dataset, batch_size=16, shuffle=False, pin_memory=True)
    test_loader_cam = DataLoader(test_dataset, batch_size=1, shuffle=False, pin_memory=True)

    print(f"[INFO] Running MC Dropout (T={mc_passes}) on the test data ...")
    mc_probs, _ = mc_dropout_predict(model, test_loader_mc, mc_passes=mc_passes)
    unc = decompose_uncertainty(mc_probs)
    mean_probs = mc_probs.mean(axis=1)
    preds = np.argmax(mean_probs, axis=1)
    confidences = np.max(mean_probs, axis=1)
    image_paths = test_dataset.df["path"].values
    print(f"[INFO] Uncertainty computed for {len(image_paths)} images.")

    print(f"[INFO] Generating {mc_passes} Grad-CAM passes per image. This may take a while ...")
    gradcam = GradCAM(model, target_layer=model.layer4)
    rows = []
    total_images = len(test_loader_cam)

    for i, (images, true_lbl) in enumerate(test_loader_cam):
        images = images.to(device)
        true_lbl, pred_lbl, img_path = true_lbl.item(), preds[i], image_paths[i]

        img_id = os.path.splitext(os.path.basename(img_path))[0]
        img_dir = os.path.join(base_dir, img_id)
        os.makedirs(img_dir, exist_ok=True)

        orig_img = images.squeeze().permute(1, 2, 0).cpu().numpy()
        orig_img = np.clip(np.array(IMAGENET_STD) * orig_img + np.array(IMAGENET_MEAN), 0, 1)
        orig_path = os.path.join(img_dir, "original.png")
        plt.imsave(orig_path, orig_img)

        raw_cams = generate_mc_gradcams(gradcam, images, target_class=pred_lbl, mc_passes=mc_passes)
        mean_path = os.path.join(img_dir, "mean_gradcam.png")
        std_path = os.path.join(img_dir, "std_gradcam.png")
        save_gradcam_image(images, raw_cams.mean(axis=0), mean_path)
        save_gradcam_image(images, raw_cams.std(axis=0), std_path)

        pass_files = []
        for p in range(mc_passes):
            pass_file = f"{p:02d}.png"
            save_gradcam_image(images, raw_cams[p], os.path.join(img_dir, pass_file))
            pass_files.append(pass_file)

        rows.append({
            "image_id": img_id,
            "image_path": img_path,
            "true_label_name": CLASS_NAMES[true_lbl],
            "pred_label_name": CLASS_NAMES[pred_lbl],
            "is_correct": int(pred_lbl == true_lbl),
            "confidence": float(confidences[i]),
            "total_uncertainty": float(unc["total"][i]),
            "aleatoric_uncertainty": float(unc["aleatoric"][i]),
            "epistemic_uncertainty": float(unc["epistemic"][i]),
            "gradcam_folder": img_dir,
            "original_image_path": orig_path,
            "mean_gradcam_path": mean_path,
            "std_gradcam_path": std_path,
            "mc_passes_folder_content": ", ".join(pass_files),
        })
        if (i + 1) % 10 == 0 or (i + 1) == total_images:
            print(f"   [Progress] Processed {i + 1}/{total_images} images ...")
    gradcam.remove_hooks()

    df_analysis = pd.DataFrame(rows)
    csv_filename = f"test_analysis_{model_name}_{seed}_mc_gradcam.csv"
    df_analysis.to_csv(csv_filename, index=False)
    print(f"\n[SUCCESS] Analysis saved to: {csv_filename}")
    print(f"[SUCCESS] Grad-CAM folders saved in: {base_dir}/")

    print("\n" + "-" * 50)
    print("SUMMARY STATISTICS (TEST SET):")
    print("-" * 50)
    print(f"Overall accuracy       : {df_analysis['is_correct'].mean() * 100:.2f}%")
    print(f"Mean confidence        : {df_analysis['confidence'].mean():.4f}")
    print(f"Mean total uncertainty : {df_analysis['total_uncertainty'].mean():.4f}")
    print(f"Mean aleatoric unc.    : {df_analysis['aleatoric_uncertainty'].mean():.4f}")
    print(f"Mean epistemic unc.    : {df_analysis['epistemic_uncertainty'].mean():.4f}")
    correct, incorrect = df_analysis["is_correct"] == 1, df_analysis["is_correct"] == 0
    print("\nEpistemic uncertainty, correct vs. incorrect predictions:")
    print(f"  Correct   : {df_analysis.loc[correct, 'epistemic_uncertainty'].mean():.4f}")
    print(f"  Incorrect : {df_analysis.loc[incorrect, 'epistemic_uncertainty'].mean():.4f}")
    print("-" * 50)
    return df_analysis

In [ ]:
# Heavy step: 50 Grad-CAM passes (plus image export) for every test image.
df_gradcam_analysis = run_test_analysis_and_mc_gradcam()

### 3.9.3 Applying the fixed validation threshold to the test set

The threshold found on the validation set in 3.9.1 is applied **without re-tuning** to the epistemic uncertainties of the test images. A prediction is accepted if its epistemic uncertainty is ≤ the threshold. The reported value is **0.00161**; if Section 3.9.1 is re-run and prints a different threshold, update `FIXED_EPISTEMIC_THRESHOLD` accordingly.

Reported: global accuracy, coverage, selective accuracy (accepted samples) and accuracy of the rejected samples. Three figures use the same bin edges for correct and incorrect predictions:

1. **Log-scale histogram** – makes the small number of incorrect predictions visible, with counts on the red bars.
2. **Dual subplot** – correct and incorrect predictions in separate panels.
3. **Zoom** – the region around the threshold.

**Input:** `test_analysis_resnet18_360_mc_gradcam.csv`. **Outputs:** `test_analysis_resnet18_360_FIXED_THRESHOLD.csv` (adds the column `is_accepted`), `test_distribution_log_scale.png`, `test_distribution_dual_subplot.png`, `test_distribution_zoom_threshold.png`, printed results.

In [ ]:
FIXED_EPISTEMIC_THRESHOLD = 0.00161   # threshold selected on the validation set (Section 3.9.1)


def apply_fixed_threshold_to_test(csv_path=f"test_analysis_{FOCUS_MODEL}_{FOCUS_SEED}_mc_gradcam.csv",
                                  fixed_threshold=FIXED_EPISTEMIC_THRESHOLD, bins_count=150):
    print("\n" + "=" * 70)
    print(f"APPLYING THE FIXED THRESHOLD: {fixed_threshold:.5f} (SELECTED ON VALIDATION)")
    print("=" * 70)
    try:
        df = pd.read_csv(csv_path)
        print(f"[INFO] Loaded {len(df)} rows from {csv_path}")
    except FileNotFoundError:
        print(f"[ERROR] File {csv_path} not found.")
        return None

    df["is_accepted"] = (df["epistemic_uncertainty"] <= fixed_threshold).astype(int)
    accepted_df, rejected_df = df[df["is_accepted"] == 1], df[df["is_accepted"] == 0]
    global_acc = df["is_correct"].mean() * 100
    coverage = df["is_accepted"].mean() * 100
    selective_acc = accepted_df["is_correct"].mean() * 100 if len(accepted_df) > 0 else 0.0
    rejected_acc = rejected_df["is_correct"].mean() * 100 if len(rejected_df) > 0 else 0.0

    print("\n" + "-" * 60)
    print("THRESHOLD EVALUATION ON THE TEST SET:")
    print("-" * 60)
    print(f"Global accuracy (no rejection)    : {global_acc:.2f}%")
    print("-" * 60)
    print(f"Threshold used                    : {fixed_threshold:.5f}")
    print(f"Coverage (accepted data)          : {coverage:.2f}%  ({len(accepted_df)} samples)")
    print(f"Selective accuracy (accepted)     : {selective_acc:.2f}%")
    print(f"Accuracy of rejected samples      : {rejected_acc:.2f}%  ({len(rejected_df)} samples)")
    print("-" * 60)

    output_csv = f"test_analysis_{FOCUS_MODEL}_{FOCUS_SEED}_FIXED_THRESHOLD.csv"
    df.to_csv(output_csv, index=False)
    print(f"\n[SUCCESS] Dataframe with column 'is_accepted' saved to: {output_csv}")

    # x-axis limit follows the data (maximum + 5 % padding)
    max_unc = df["epistemic_uncertainty"].max()
    xlim_max = max_unc * 1.05
    print(f"\n[INFO] Maximum epistemic uncertainty: {max_unc:.5f}")
    print(f"[INFO] Right x-limit of the plots: {xlim_max:.5f}")

    correct_data = df.loc[df["is_correct"] == 1, "epistemic_uncertainty"]
    incorrect_data = df.loc[df["is_correct"] == 0, "epistemic_uncertainty"]
    edges = np.linspace(0, xlim_max, bins_count + 1)          # common bin edges for both groups
    bin_width = edges[1] - edges[0]
    ticks = np.linspace(0, xlim_max, 6)

    def annotate_counts(ax, counts, color, fontsize, y_factor=None, step=1, min_count=0):
        for i in range(0, len(counts), step):
            if counts[i] > min_count:
                y = counts[i] * y_factor if y_factor else counts[i]
                ax.text(edges[i] + bin_width / 2, y, f"{int(counts[i])}", ha="center", va="bottom",
                        fontsize=fontsize, color=color, fontweight="bold")

    # ---- Figure 1: log scale ----
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.hist(correct_data, bins=edges, alpha=0.7, label=f"Correct (n={len(correct_data)})",
            color="green", edgecolor="darkgreen", linewidth=0.5)
    inc_counts, _, _ = ax.hist(incorrect_data, bins=edges, alpha=0.9, label=f"Incorrect (n={len(incorrect_data)})",
                               color="red", edgecolor="darkred", linewidth=1.0)
    ax.axvline(x=fixed_threshold, color="black", linestyle="--", linewidth=2.5,
               label=f"Threshold ({fixed_threshold:.5f})")
    ax.set_yscale("log")
    ax.set_ylim(bottom=0.5)
    annotate_counts(ax, inc_counts, "darkred", 9, y_factor=1.5)
    ax.set_xlim(0, xlim_max)
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"{x:.5f}" for x in ticks], fontsize=10)
    ax.set_xlabel("Epistemic Uncertainty", fontsize=12, fontweight="bold")
    ax.set_ylabel("Sample Count (Log Scale)", fontsize=12, fontweight="bold")
    ax.set_title("Epistemic Uncertainty Distribution (Log Scale)\nMakes Small Counts Visible",
                 fontsize=14, fontweight="bold", pad=15)
    ax.legend(loc="upper right", fontsize=10, framealpha=0.9)
    ax.grid(True, linestyle=":", alpha=0.7, which="both")
    plt.tight_layout()
    plt.savefig("test_distribution_log_scale.png", dpi=300, bbox_inches="tight")
    print("[SUCCESS] Log-scale plot saved as: test_distribution_log_scale.png")
    plt.show()

    # ---- Figure 2: dual subplot ----
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    cor_counts, _, _ = ax1.hist(correct_data, bins=edges, alpha=0.7, color="green", edgecolor="darkgreen", linewidth=0.5)
    ax1.axvline(x=fixed_threshold, color="black", linestyle="--", linewidth=2)
    ax1.set_ylabel("Correct Predictions Count", fontsize=11, fontweight="bold")
    ax1.set_title(f"Correct Predictions Distribution (n={len(correct_data)})", fontsize=12, fontweight="bold")
    ax1.grid(True, linestyle=":", alpha=0.7)
    ax1.set_xlim(0, xlim_max)
    annotate_counts(ax1, cor_counts, "darkgreen", 8, step=20, min_count=10)

    inc_counts2, _, _ = ax2.hist(incorrect_data, bins=edges, alpha=0.9, color="red", edgecolor="darkred", linewidth=1.0)
    ax2.axvline(x=fixed_threshold, color="black", linestyle="--", linewidth=2)
    ax2.set_xlabel("Epistemic Uncertainty", fontsize=12, fontweight="bold")
    ax2.set_ylabel("Incorrect Predictions Count", fontsize=11, fontweight="bold")
    ax2.set_title(f"Incorrect Predictions Distribution (n={len(incorrect_data)})", fontsize=12, fontweight="bold")
    ax2.grid(True, linestyle=":", alpha=0.7)
    ax2.set_xlim(0, xlim_max)
    annotate_counts(ax2, inc_counts2, "darkred", 10)
    ax2.set_xticks(ticks)
    ax2.set_xticklabels([f"{x:.5f}" for x in ticks], fontsize=10)
    plt.tight_layout()
    plt.savefig("test_distribution_dual_subplot.png", dpi=300, bbox_inches="tight")
    print("[SUCCESS] Dual-subplot figure saved as: test_distribution_dual_subplot.png")
    plt.show()

    # ---- Figure 3: zoom around the threshold ----
    zoom_range = min(0.005, xlim_max * 0.25)                   # 25 % of the x-range, at most 0.005
    zoom_edges = np.linspace(0, zoom_range, 51)
    zoom_width = zoom_edges[1] - zoom_edges[0]
    correct_zoom = correct_data[correct_data <= zoom_range]
    incorrect_zoom = incorrect_data[incorrect_data <= zoom_range]

    fig, ax = plt.subplots(figsize=(14, 7))
    ax.hist(correct_zoom, bins=zoom_edges, alpha=0.7, label=f"Correct (n={len(correct_zoom)})",
            color="green", edgecolor="darkgreen", linewidth=0.5)
    inc_counts_z, _, _ = ax.hist(incorrect_zoom, bins=zoom_edges, alpha=0.9, label=f"Incorrect (n={len(incorrect_zoom)})",
                                 color="red", edgecolor="darkred", linewidth=1.0)
    ax.axvline(x=fixed_threshold, color="black", linestyle="--", linewidth=2.5,
               label=f"Threshold ({fixed_threshold:.5f})")
    for i, count in enumerate(inc_counts_z):
        if count > 0:
            ax.text(zoom_edges[i] + zoom_width / 2, count * 1.1, f"{int(count)}", ha="center", va="bottom",
                    fontsize=9, color="darkred", fontweight="bold")
    ax.set_xlim(0, zoom_range)
    zoom_ticks = np.linspace(0, zoom_range, 6)
    ax.set_xticks(zoom_ticks)
    ax.set_xticklabels([f"{x:.5f}" for x in zoom_ticks], fontsize=10)
    ax.set_xlabel("Epistemic Uncertainty", fontsize=12, fontweight="bold")
    ax.set_ylabel("Sample Count", fontsize=12, fontweight="bold")
    ax.set_title(f"Zoomed View: 0 to {zoom_range:.5f}\nFocus on Threshold Area", fontsize=14, fontweight="bold", pad=15)
    ax.legend(loc="upper right", fontsize=10, framealpha=0.9)
    ax.grid(True, linestyle=":", alpha=0.7)
    plt.tight_layout()
    plt.savefig("test_distribution_zoom_threshold.png", dpi=300, bbox_inches="tight")
    print("[SUCCESS] Zoom plot saved as: test_distribution_zoom_threshold.png")
    plt.show()
    return df

In [ ]:
df_fixed_threshold = apply_fixed_threshold_to_test()

# Appendix A — Catalogue of outputs

Everything the notebook prints, saves or draws, in execution order. *Console* = printed in the notebook; *Figure* = shown inline and (if a file name is given) saved; *File* = written to the working directory.

| Section | Output | Type | Description |
|---|---|---|---|
| 0.2 | `Using device: …` | Console | CPU or GPU that is used |
| 1.1 | dataset scan counts, class × folder table | Console | number of images found in `Training` / `Testing` per class; warnings for missing folders |
| 1.2 | split summary, class × subset table | Console | sizes of train / validation / test and the class balance of each (stratification check) |
| 1.3 | `augmented_train/<id>_aug0k_<label>.png` | File | 3 augmented PNG copies of every training image |
| 1.3 | `df_train.csv` | File | original + augmented training rows, shuffled (columns `id`, `label`, `original`, `path`) |
| 1.3 | `df_validation.csv`, `df_test.csv` | File | validation and test tables (same columns) |
| 1.3 / 1.4 | final-dataset summary, `head()` of the three tables | Console | row counts (original + augmented) and a preview |
| 2.2 / 2.3 | Optuna log per trial | Console | trial number, validation accuracy and sampled hyper-parameters; pruned trials are marked |
| 2.3 | `df_report_<model>.csv` (×6) | File | one row per trial: validation / test accuracy, ROC-AUC, hyper-parameters, epochs run, duration, status (`COMPLETE` / `PRUNED`) |
| 2.4 | `optuna_6models_combined_report.xlsx` | File | all trials of the six models with `global_trial_id`, sorted by validation accuracy |
| 2.4 | trial count, status counts, top-3 table | Console | how many trials completed / were pruned; best three completed trials per backbone |
| 3.1.2 | one line per run | Console | model, seed, train / validation / test accuracy, test ECE, run time |
| 3.1.2 | `results_30_seeds_<model>.xlsx` (×6) | File | 30 rows per backbone: hyper-parameters, accuracy, ROC-AUC, ECE, MCE, Brier, NLL, AURC, macro F1 / precision / recall / specificity, duration, checkpoint path |
| 3.1.2 | `saved_models/<model>/<model>_seed_<seed>.pth` | File | 180 fine-tuned weight files |
| 3.1.3 | `results_180_runs_all_models.xlsx` | File | 8 sheets: `raw_180_runs`, `statistical_summary`, `mean_std_formatted`, `mean_only`, `std_only`, `max_only`, `min_only`, `composite_score_ranking` |
| 3.1.3 | CDSS ranking table | Console | rank, total score and the three pillar scores of each backbone |
| 3.1.4 | normality summary table | Console | mean accuracy, Shapiro–Wilk p-value and the verdict (normal / not normal) per model |
| 3.1.4 | `accuracy_normality_analysis.xlsx` | File | 2 sheets: `shapiro_summary_accuracy` (N, mean, SD, min, max, W, p-value, conclusion per model) and `raw_30_seed_accuracy` (test accuracy, seeds × models) |
| 3.1.4 | `normality_plots/accuracy_normality_<model>.png` (×6) | Figure | histogram with KDE and Q–Q plot of the 30 test accuracies of each model |
| 3.1.4 | `normality_plots/accuracy_boxplot_all_models.png` | Figure | box plot (mean marked) of the 30 test accuracies of all models |
| 3.1.5 | Friedman and post-hoc results | Console | Friedman χ² and p-value; the 15 pairwise Wilcoxon tests with raw and Holm-adjusted p-values and the verdict |
| 3.1.5 | `results_advanced_statistical_tests.xlsx` | File | 3 sheets: `friedman_test`, `posthoc_wilcoxon_holm`, `raw_30_seed_accuracy` |
| 3.1.6 | paired-test table | Console | Shapiro p-value, test used (t-test or Wilcoxon), statistic, p-value, mean difference, significance per metric |
| 3.1.6 | `statistical_comparison_no_duration.xlsx` | File | the same table as Excel |
| 3.1.6 | `model_comparison_performance_only.png` | Figure | box plots of ResNet18 vs. DenseNet121 for accuracy, ROC-AUC, macro F1 and ECE with the p-value in each title |
| 3.2 | baseline table; A / B / C tables | Console | pillar scores; rank distribution under random weights, Kendall τ, scenario rankings, share of the simplex per model, seed-bootstrap rank distribution and 95 % intervals; AURC warning / consistency check |
| 3.2 | `mcda_sensitivity_results.xlsx` | File | 9 sheets: `baseline`, `dirichlet_rank_dist`, `dirichlet_kendall_tau`, `dirichlet_pairwise`, `weight_scenarios`, `simplex_area`, `bootstrap_baseline_w`, `bootstrap_ci`, `bootstrap_joint` |
| 3.2 | `mcda_sensitivity.png` | Figure | (a) stacked rank distribution under random weights; (b) top-ranked model over the weight simplex |
| 3.3.2 | one line per (model, seed) | Console | validation / test accuracy while extracting uncertainty scores |
| 3.3.2 | `uncertainty_cache/<model>_seed_<seed>.npz` | File | stored labels, mean probabilities and the three uncertainty scores for validation and test (resumable cache) |
| 3.3.2 | selective-prediction table | Console | coverage, selective accuracy, rejected errors and error-rejection rate (mean ± SD over seeds) at 90 % target coverage |
| 3.3.2 | AUROC table | Console | error-detection AUROC with 95 % bootstrap interval per model and score |
| 3.3.2 | `results_selective_prediction_30seeds.xlsx` | File | 6 sheets: `selective_per_seed`, `selective_mean_sd`, `selective_numeric`, `auroc_per_seed`, `auroc_ci_bootstrap`, `auroc_score_differences` |
| 3.4 | per-seed lines, mean ± SD per mode | Console | all metrics for the deterministic, standard-dropout and MC-Dropout modes |
| 3.4 | `three_mode_comparison_resnet18.xlsx` | File | 90 rows (30 seeds × 3 modes) with all metrics |
| 3.5 | Wilcoxon results | Console | mean ± SD, statistic and p-value for ECE, NLL and Brier (deterministic vs. MC Dropout) |
| 3.6 | convergence summary table | Console | mean ± SD of accuracy, entropy, variance and ECE for every T |
| 3.6 | `mc_dropout_convergence_summary.csv` | File | the same table as CSV |
| 3.6 | `mc_dropout_convergence.png` | Figure | four panels (accuracy, entropy, variance, ECE vs. T) with ±1 SD bands |
| 3.7 | `plots/resnet18_seed360_per_class_metrics.png` | Figure | grouped bars: precision, recall, F1, accuracy, specificity, AUC-ROC per class |
| 3.7 | `plots/resnet18_seed360_confusion_matrix.png` | Figure | confusion matrix (counts) of the seed-360 run |
| 3.7 | `plots/resnet18_seed360_reliability_diagram.png` | Figure | confidence-vs-accuracy curve with bin counts and ECE |
| 3.8 | `resnet18_seed360_uncertainty_distribution_per_class.png` | Figure | box plots of total / aleatoric / epistemic uncertainty per true class |
| 3.8 | `resnet18_seed360_uncertainty_correct_vs_wrong_analysis.png` | Figure | entropy-vs-confidence scatter and aleatoric / epistemic histograms for correct vs. wrong predictions |
| 3.8 | per-class uncertainty summary | Console | counts and mean ± SD of the three uncertainty measures per class |
| 3.9.1 | threshold result block | Console | selected threshold, coverage, selective accuracy, F1 and global accuracy on validation |
| 3.9.1 | `epistemic_threshold_analysis.png` | Figure | uncertainty histogram with the threshold; coverage-vs-selective-accuracy curve |
| 3.9.1 | `threshold_search_log_resnet18_360.csv` | File | coverage, selective accuracy and F1 for each of the 99 candidate thresholds |
| 3.9.2 | `gradcam_outputs/resnet18_360_test/<image_id>/…` | File | `original.png`, `mean_gradcam.png`, `std_gradcam.png`, and 50 single-pass maps `00.png … 49.png` for every test image |
| 3.9.2 | `test_analysis_resnet18_360_mc_gradcam.csv` | File | per test image: true / predicted class, correctness, confidence, total / aleatoric / epistemic uncertainty, file paths |
| 3.9.2 | summary statistics | Console | accuracy, mean confidence and uncertainties; epistemic uncertainty of correct vs. incorrect predictions |
| 3.9.3 | threshold evaluation on test | Console | global accuracy, coverage, selective accuracy and accuracy of the rejected samples |
| 3.9.3 | `test_analysis_resnet18_360_FIXED_THRESHOLD.csv` | File | the Grad-CAM table plus the accept / reject flag `is_accepted` |
| 3.9.3 | `test_distribution_log_scale.png`, `test_distribution_dual_subplot.png`, `test_distribution_zoom_threshold.png` | Figure | epistemic-uncertainty distributions of correct vs. incorrect predictions with the threshold line |

# Appendix B — Revision notes

*These notes document what was changed while tidying the notebook and what should be verified before submission. They can be deleted from the published version.*

**Corrections that fix errors (the original notebook would crash or give wrong values)**

1. **Augmentation cell** used `T.Compose(...)` although only `transforms` was imported → `NameError`. Now uses `transforms`.
2. **Summary cell of the Optuna section** (now 2.4, originally 2.3) was indented as a whole → `IndentationError`. Fixed; the hard-coded "150 trials" message is now computed.
3. **Training CSV name:** the augmentation cell saved `df_train_Final.csv`, but every later cell (and its own summary message) reads `df_train.csv`. It now saves `df_train.csv`.
4. **Checkpoints were never saved** by the 30-seed function, although Sections 3.3 – 3.9 load `saved_models/<model>/<model>_seed_<seed>.pth` and Section 3.3 reads a `model_path` column. The function now saves each `state_dict` and writes `model_path`.
5. **AURC in Sections 3.4 / 3.5** used an older, incorrect implementation (numerators and denominators of the risk did not belong to the same coverage level; values could even exceed 1), while the MC-Dropout column came from the corrected version used in 3.1. All sections now use the single corrected `compute_risk_coverage` of Section 0.6. The Wilcoxon tests (ECE, NLL, Brier) are not affected, but the AUCR values printed for the *deterministic* and *dropout* modes change.
6. **Section 3.8** rebuilt the model with `dropout_rate = 0.5`, whereas ResNet18 was trained and evaluated with 0.625 (Sections 3.9.x already used 0.625). It now takes the rate from `BEST_CONFIGS`, so the uncertainty values of Figure 3.8 change.
7. **Reliability diagram (3.7)** drew empty confidence bins as points with accuracy 0, creating false dips in the curve; empty bins are now skipped.
8. **MCDA figure (3.2)** called `matplotlib.use("Agg")`, which disables inline plotting for every later cell; removed (the figure is now also shown inline).

**Changes that improve reproducibility (new random streams or orderings, same method)**

* File names are sorted before the stratified split, and the augmentation is seeded (`SPLIT_SEED = 42`), so Section 1 gives the same result on any machine. On a machine whose `os.listdir` order differed, a re-run can produce a different split from the CSV files you already have.
* The standard-dropout mode of 3.4 now calls `set_seed(seed)` first, so it is repeatable.
* The Optuna search seed is printed (it is time-based by design).
* Histograms that overlay correct and incorrect predictions (3.9.1, 3.9.3) now use shared bin edges; the 3.8 histograms keep the original ranges but are extended if data exceed them; the histogram axis label "Density" was corrected to "Frequency" because counts are plotted.

**Structural tidy-up (no change in method)**

* Dataset class, MC-Dropout layer, model factory, metrics and the uncertainty decomposition are defined once (Section 0) instead of up to eight times; `ToggleableDropout` was merged into `MCDropout(enabled=…)`.
* Duplicated cells (the threshold search and the Grad-CAM cell each appeared twice), eight empty cells and the three bare dataframe cells were removed or merged; Sections 3.4 and 3.5 no longer repeat the whole inference.
* Numbering: Section 1 now has four steps (1.1 scan, 1.2 split, 1.3 augmentation and export, 1.4 preview) and Section 2 four (2.1 – 2.4); Sections 3.1 – 3.9 keep their original numbers, with new sub-sections 3.1.1 – 3.1.4, 3.3.1 – 3.3.2 and 3.9.1 – 3.9.3.
* Per-model run cells were replaced by loops over `BEST_CONFIGS`; the Optuna runs skip models whose report already exists (`FORCE_RERUN_HPO`).
* `if __name__ == "__main__"` blocks and the `argparse` interface of the MCDA script were replaced by explicit function calls.
* Section headings were corrected ("Preprocessing", "Reliability diagram", "MC Dropout") and the unlabelled paired-test cell became 3.1.6 (it now follows the normality test of 3.1.4 and the Friedman / post-hoc tests of 3.1.5). Original heading *"Finding the three best configurations out of 6 models"* was changed to describe what the code does (a search for all six backbones; all six continue to Section 3).
* Indonesian comments, messages and column / sheet names in the code were translated, and **all remaining Indonesian file and folder names were renamed as well** (table below). The conversion cell in Section 0.8 applies the same renaming to files produced by earlier runs, including the image paths stored in `df_train.csv` and the sheet / column names inside existing Excel files.

| Former name | New name | Where |
|---|---|---|
| `df_validasi.csv` | `df_validation.csv` | file |
| `augmentasi_train/` | `augmented_train/` | folder; also the `path` column of `df_train.csv` |
| `hasil_30_seeds_<model>.xlsx` | `results_30_seeds_<model>.xlsx` | file (×6) |
| `hasil_180_runs_semua_model.xlsx` | `results_180_runs_all_models.xlsx` | file |
| `hasil_selective_prediction_30seeds.xlsx` | `results_selective_prediction_30seeds.xlsx` | file |
| `perbandingan_3_mode_resnet18.xlsx` | `three_mode_comparison_resnet18.xlsx` | file |
| `ringkasan_statistik` | `statistical_summary` | sheet of `results_180_runs_all_models.xlsx` |
| `auroc_selisih_skor` | `auroc_score_differences` | sheet of `results_selective_prediction_30seeds.xlsx` |
| `perbandingan` | `comparison` | column of sheet `auroc_score_differences` |
| `ci_tidak_memuat_0` | `ci_excludes_zero` | column of sheet `auroc_score_differences` |
| `auroc_sd_antar_seed` | `auroc_sd_across_seeds` | column of sheet `auroc_ci_bootstrap` |

**Statistical scripts added later (3.1.4 and 3.1.5)**

* The two scripts (Shapiro–Wilk normality of the accuracy, and Friedman + pairwise Wilcoxon with Holm) were translated like the rest (file, sheet, column and value names; sheet names follow the lower-case snake_case style of the other workbooks) and now use the shared imports and file names.
* *Holm correction:* the original multiplied each sorted p-value by $(m - \text{rank})$ but did not enforce monotonicity, so a larger raw p-value could receive a smaller adjusted p-value. `holm_adjust` takes the running maximum (and caps at 1). The significance decision uses the unrounded values, and the p-values are stored unrounded (rounded to 5 decimals, tiny p-values appear as 0).
* *Complete blocks:* the Friedman test uses only the seeds for which all models have a value; unequal vector lengths would otherwise stop the test.
* *Plot style:* `sns.set_theme` / `rcParams` were moved into a context manager so that they no longer change the style of every later figure.
* Files written by the earlier Indonesian-named versions of these scripts (`analisis_normalitas_akurasi.xlsx`, `hasil_uji_statistik_lanjutan.xlsx`, `grafik_normalitas/`) are **not** converted by Section 0.8 (their sheet names, column names and values are Indonesian); re-run 3.1.4 and 3.1.5, which is fast, and delete the old files.

**Items to verify or state in the manuscript**

* The configurations in `BEST_CONFIGS` are typed by hand (three significant digits) from the Optuna reports; the rule used to select them (e.g. highest validation accuracy) is not encoded in the notebook.
* The criterion for choosing **seed 360** for Sections 3.7 – 3.9 is not encoded in the notebook.
* Preprocessing differs between the search (grayscale → 3 channels, Section 2) and the final runs (RGB, Section 3). For MRI JPEGs with identical channels this should be equivalent, but it is worth confirming.
* Standard deviations: sample SD (`ddof = 1`) in 3.1 – 3.4, population SD (`ddof = 0`) in 3.5 and 3.6.
* Significance in 3.1.6 uses $p<0.05$ for four metrics without multiple-comparison correction; the Shapiro–Wilk tests of 3.1.4 are one test per backbone, also uncorrected, and have limited power with 30 seeds.
* Sections 3.1.5 and 3.1.6 treat the seed index as a matching variable. The same seed value is used for every backbone, but the architectures and batch sizes differ, so the runs are matched only nominally. State this assumption in the manuscript, or add an unpaired check (Kruskal–Wallis / Mann–Whitney) as a sensitivity analysis.
* The fixed threshold 0.00161 of 3.9.3 is a typed constant; MC Dropout is stochastic, so a re-run of 3.9.1 may give a slightly different value.